In [1]:
import tensorflow as tf
print(tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1786367649.694053    8505 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786367650.053748    8505 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786367651.419180    8505 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ==============================================================================
# RA (SOURCE) -> AML from MILE/GSE13159 (EXTERNAL TARGET, FREEZE & ADAPT)
# TOP 116 ROBUST KEGG PATHWAYS | BONE MARROW TISSUE-MATCHED
# LOCAL WINDOWS PYTHON VERSION
#
# PURPOSE: Independent second external validation of RA->AML transfer,
# using MILE cohort AML (bone marrow) samples -- separate from GSE15061.
#
# PROTOCOL (identical logic to CML/ALL/MDS Freeze & Adapt runs):
#   - eta (pathway contribution) and bias FROZEN from RA source training
#   - shared_projection (w) ADAPTED on AML target-TRAIN fold only
#   - Evaluated on AML target-TEST fold (held out, never touched for adaptation)
#   - Youden J threshold computed from target-TRAIN fold only (no leakage)
#   - Stratified 5-fold CV on target domain, repeated across REPEATS
# ==============================================================================

import os, sys, gc, random, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

required = {"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn",
            "openpyxl":"openpyxl","tensorflow":"tensorflow","gseapy":"gseapy"}
for imp, pipn in required.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pipn])

import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed
from gseapy import get_library

# ------------------------------------------------------------------
# 0) SETTINGS -- kendi klasörünüze göre BASE_DIR'i düzenleyin
# ------------------------------------------------------------------
SEED = 42
BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"

MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"

TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi.xlsx"

SAVE_PATH = BASE_DIR / "RA2AML_MILE_FREEZE_ADAPT_TOP116"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

LIBRARY = "KEGG_2021_Human"
TOP_N = 116
REPEATS = 80            # hız için düşürüldü (CML'de 15-20 civarı zaten stabildi)
N_SPLITS = 2
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400       # 50'den düşürüldü, adaptasyon daha hızlı
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20
TARGET_DISEASE = "AML"

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
set_random_seed(SEED); clear_session()

print("="*100)
print(f"RA -> {TARGET_DISEASE} (MILE) | FREEZE & ADAPT | TOP {TOP_N} | REPEATS={REPEATS}")
print("="*100)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def pick_col(df, cands):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in cands:
        k = str(c).strip().lower()
        if k in lookup: return lookup[k]
    return None

def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA","NAN","NONE","---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)

def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name","Gene","Genes","gene","gene_symbol","SYMBOL","X","Unnamed: 0"])
    if gene_col is None: gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))

def parse_field(value, field):
    if pd.isna(value): return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None

def map_main_label(cls):
    if cls is None or pd.isna(cls): return None
    c = str(cls).strip().upper()
    if c.startswith("AML"): return "AML"
    if c == "CLL": return "CLL"
    if c == "CML": return "CML"
    if c == "MDS": return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c: return "HE"
    if "ALL" in c: return "ALL"
    return "OTHER"

def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None: X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2/(2.0*sigma**2)).astype(np.float32)

def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0: return 1.0
    s = float(np.mean(np.sqrt(upper)))
    return s if np.isfinite(s) and s > 0 else 1.0

def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    proj = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(proj + [bias])
    out = Dense(1, activation="sigmoid", use_bias=False,
                kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model

def configure_target_adaptation(model):
    # CRITICAL: eta (final_output) and bias stay FROZEN, consistent with CML protocol
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model

def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]

def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, w)}

def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(spec),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }

def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2: return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values)-1)
    return (float(mean - tcrit*sem), float(mean + tcrit*sem))

# ------------------------------------------------------------------
# INPUT CHECK
# ------------------------------------------------------------------
print("\n[INPUT CHECK]")
for label, path in {"RA expression":RA_EXPR, "RA phenotype":RA_PHENO,
                     "MILE expression":MILE_EXPR, "MILE phenotype":MILE_FULL_PHENO,
                     "Pathway file":TOP_PATH_FILE}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists(): raise FileNotFoundError(path)

# ------------------------------------------------------------------
# 1) LOAD RA
# ------------------------------------------------------------------
print("\n[1/7] LOADING RA")
expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")
ra_sample_col = pick_col(ph_ra, ["sample","Sample","GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw","group","label"])
if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA","HE"])].copy()
ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]
ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x=="RA" else 0 for x in ra_group.values], dtype=int)
print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full==1))} | HE={int(np.sum(y_ra_full==0))}")

# ------------------------------------------------------------------
# 2) LOAD MILE
# ------------------------------------------------------------------
print("\n[2/7] LOADING MILE / GSE13159")
expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")
mile_sample_col = pick_col(ph_mile, ["GSM","sample","Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])
if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()
avail = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(avail)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

aml_subtypes = ph_mile.loc[ph_mile["main_label"]=="AML", "leukemia_class"].value_counts()
print("\nAML leukemia_class breakdown (subtypes merged as single AML label):")
print(aml_subtypes)

# ------------------------------------------------------------------
# 3) BUILD AML BONE-MARROW TARGET
# ------------------------------------------------------------------
print(f"\n[3/7] BUILDING {TARGET_DISEASE} BONE-MARROW TARGET (MILE cohort)")
is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)
he_bm = ph_mile.loc[(ph_mile["main_label"]=="HE") & is_bm, mile_sample_col].tolist()
case_bm = ph_mile.loc[(ph_mile["main_label"]==TARGET_DISEASE) & is_bm, mile_sample_col].tolist()
print(f"{TARGET_DISEASE} bone marrow = {len(case_bm)} | Healthy bone marrow = {len(he_bm)}")
if len(case_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid AML/healthy bone-marrow target set found.")

target_samples_all = he_bm + case_bm
y_target_all = np.array([0]*len(he_bm) + [1]*len(case_bm), dtype=int)
print("Target total:", len(target_samples_all), "| AML prevalence:", round(float(np.mean(y_target_all)),4))

print("\nNOTE: MILE (GSE13159) AML samples are INDEPENDENT of the manuscript's")
print("primary AML target dataset (GSE15061). This is a second, separate")
print("external validation cohort, not a duplicate of the main result.")

# ------------------------------------------------------------------
# 4) COMMON GENES
# ------------------------------------------------------------------
print("\n[4/7] COMMON GENES")
common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]
print("Common genes:", len(common_genes))

# ------------------------------------------------------------------
# 5) LOAD TOP 116 PATHWAYS
# ------------------------------------------------------------------
print("\n[5/7] LOADING TOP 116 ROBUST KEGG PATHWAYS")
freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
pcol = pick_col(freq, ["Pathway"]); ccol = pick_col(freq, ["Selection_Count","Count"])
if pcol is None or ccol is None:
    raise KeyError(f"Pathway-frequency columns not found: {list(freq.columns)}")
top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()
kegg = get_library(name=LIBRARY, organism="Human")
active_paths = []
for p in top_paths:
    if p in kegg:
        genes = sorted(set(kegg[p]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((p, genes))
print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")
if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top116_Pathways.xlsx", index=False)

# ------------------------------------------------------------------
# 6) MAIN LOOP: TRAIN RA -> FREEZE & ADAPT ON AML (5-fold CV x REPEATS)
# ------------------------------------------------------------------
print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON {TARGET_DISEASE} "
      f"({REPEATS} repeats x {N_SPLITS} folds)")

all_rows = []
prediction_rows = []

for rep in range(REPEATS):
    rep_number = rep + 1
    split_seed = SEED + rep

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)), test_size=RA_TEST_SIZE,
        stratify=y_ra_full, random_state=split_seed
    )
    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes, columns=source_samples
    )

    Ktr_list, pathway_sigmas = [], {}
    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))
    Xtr = np.transpose(np.stack(Ktr_list), (1,0,2)).astype(np.float32)

    clear_session(); set_random_seed(split_seed)
    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(to_inputs(Xtr), y_source, epochs=EPOCHS_RA, batch_size=BATCH_RA,
                      verbose=0, class_weight=class_weights(y_source), shuffle=True)
    source_weights = source_model.get_weights()

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):
        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list, Ktest_list = [], []
        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1,0,2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1,0,2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed*1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)
        target_model.fit(to_inputs(Xtrain_t), y_train_t, epochs=EPOCHS_ADAPT,
                          batch_size=BATCH_ADAPT, verbose=0,
                          class_weight=class_weights(y_train_t), shuffle=True)

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)  # Youden J, TRAIN fold only

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        met = compute_metrics(y_test_t, p_test, thr)
        met.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        all_rows.append(met)

        for s, y, p in zip(target_test, y_test_t, p_test):
            prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": s,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session(); gc.collect()

    recent = pd.DataFrame(all_rows)
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"running mean AUROC so far = {recent['AUROC'].mean():.4f}")

    del source_model, Xtr
    clear_session(); gc.collect()

metrics_df = pd.DataFrame(all_rows)
predictions_df = pd.DataFrame(prediction_rows)

# ------------------------------------------------------------------
# 7) SUMMARY + SAVE
# ------------------------------------------------------------------
print("\n[7/7] SUMMARY")
repeat_means = metrics_df.groupby("Repeat")[["AUROC","PR_AUC","Accuracy","Balanced_Accuracy",
                                              "Precision","Recall","Specificity","F1"]].mean()

summary_rows = []
for metric in ["AUROC","PR_AUC","Accuracy","Balanced_Accuracy","Precision","Recall","Specificity","F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric, "N_repeats": len(values),
        "Mean": float(np.mean(values)), "SD": float(np.std(values, ddof=1)),
        "Median": float(np.median(values)), "Min": float(np.min(values)), "Max": float(np.max(values)),
        "CI95_low": low, "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))
print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")

out_xlsx = SAVE_PATH / f"RA2{TARGET_DISEASE}_MILE_FreezeAdapt_Top116_{REPEATS}reps.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat], "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    aml_subtypes.reset_index().to_excel(writer, sheet_name="AML_Subtype_Check", index=False)

print("\nSaved:", out_xlsx)
print("="*100)
print("DONE.")
print("="*100)

RA -> AML (MILE) | FREEZE & ADAPT | TOP 116 | REPEATS=80
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression     : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype      : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file        : True | /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi.xlsx

[1/7] LOADING RA
Reading combat_corrected_by_gse.xlsx | gene column=Gen_name
RA n=195 | RA=163 | HE=32

[2/7] LOADING MILE / GSE13159
Reading GSE13159_gene_unique.xlsx | gene column=gene

MILE label x sample type:
sample_type_norm  bone marrow  peripheral blood
main_label                                     
ALL                       710                40
AML                       501              

I0000 00:00:1786368064.280949    8505 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7814 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1786368065.612957    8784 service.cc:153] XLA service 0x7537a0041680 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786368065.612972    8784 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3080, Compute Capability 8.6 (Driver: 12.9.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1786368065.642652    8784 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1786368065.760346    8784 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1786368069.503723    8784 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  ... repeat 1/80 done | running mean AUROC so far = 0.6891
  ... repeat 2/80 done | running mean AUROC so far = 0.7531
  ... repeat 3/80 done | running mean AUROC so far = 0.7265
  ... repeat 4/80 done | running mean AUROC so far = 0.7430
  ... repeat 5/80 done | running mean AUROC so far = 0.7491
  ... repeat 6/80 done | running mean AUROC so far = 0.7583
  ... repeat 7/80 done | running mean AUROC so far = 0.7712
  ... repeat 8/80 done | running mean AUROC so far = 0.7557
  ... repeat 9/80 done | running mean AUROC so far = 0.7625
  ... repeat 10/80 done | running mean AUROC so far = 0.7672
  ... repeat 11/80 done | running mean AUROC so far = 0.7551
  ... repeat 12/80 done | running mean AUROC so far = 0.7554
  ... repeat 13/80 done | running mean AUROC so far = 0.7612
  ... repeat 14/80 done | running mean AUROC so far = 0.7608
  ... repeat 15/80 done | running mean AUROC so far = 0.7631
  ... repeat 16/80 done | running mean AUROC so far = 0.7592
  ... repeat 17/80 done | running

In [3]:
# ==============================================================================
# RA (SOURCE) -> CML from MILE/GSE13159 (EXTERNAL TARGET, FREEZE & ADAPT)
# TOP 116 ROBUST KEGG PATHWAYS | BONE MARROW TISSUE-MATCHED
# LINUX GPU VERSION WITH CHECKPOINT/RESUME
#
# PROTOCOL:
#   - RA is the source domain
#   - CML/HE bone marrow samples from MILE/GSE13159 form the target domain
#   - eta (pathway contribution) and bias are FROZEN from RA source training
#   - shared_projection (w) is ADAPTED on CML target-TRAIN fold only
#   - CML target-TEST fold remains completely held out
#   - Youden J threshold is calculated from target-TRAIN fold only
#   - Stratified CV is repeated across REPEATS
#
# CHECKPOINT: after every completed repeat, results are appended to a CSV.
# If interrupted (power loss, disconnect), re-running this script will skip
# already-completed repeats and resume from where it left off.
# ==============================================================================

import os
import sys
import gc
import random
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ==============================================================================
# 0) INSTALL REQUIRED PACKAGES IF MISSING
# ==============================================================================

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "tensorflow": "tensorflow",
    "gseapy": "gseapy"
}

for imp, pipn in required.items():
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pipn]
        )

import numpy as np
import pandas as pd
import scipy.stats as st

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

from gseapy import get_library


# ==============================================================================
# 1) SETTINGS -- Linux GPU makinesine göre ayarlandı, CML için doğru hedef
# ==============================================================================

SEED = 42

BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"

MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"

TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi.xlsx"

# IMPORTANT: separate output folder from the AML run
SAVE_PATH = BASE_DIR / "RA2CML_MILE_FREEZE_ADAPT_TOP116"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Checkpoint files -- make the run resumable after interruption
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

LIBRARY = "KEGG_2021_Human"
TOP_N = 116

REPEATS = 80
N_SPLITS = 2          # AML koşusunda hızlı çalıştığı için aynı ayar kullanıldı
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20

# IMPORTANT: this is CML, not AML
TARGET_DISEASE = "CML"


# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

set_random_seed(SEED)
clear_session()


print("=" * 110)
print(
    f"RA -> {TARGET_DISEASE} | MILE/GSE13159 | "
    f"FREEZE & ADAPT | TOP {TOP_N} | LINUX GPU"
)
print("=" * 110)

print("Python executable :", sys.executable)
print("RA expression     :", RA_EXPR)
print("RA phenotype      :", RA_PHENO)
print("MILE expression   :", MILE_EXPR)
print("MILE phenotype    :", MILE_FULL_PHENO)
print("Pathway file      :", TOP_PATH_FILE)
print("Output directory  :", SAVE_PATH)
print("REPEATS           :", REPEATS)
print("N_SPLITS          :", N_SPLITS)
print("EPOCHS_RA         :", EPOCHS_RA)
print("EPOCHS_ADAPT      :", EPOCHS_ADAPT)
print("LR_ADAPT          :", LR_ADAPT)

print(
    "Visible GPUs      :",
    tf.config.list_physical_devices("GPU")
)


# ==============================================================================
# 2) HELPER FUNCTIONS
# ==============================================================================

def pick_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = str(c).strip().lower()
        if key in lookup:
            return lookup[key]
    return None


def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA", "NAN", "NONE", "---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)


def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name", "Gene", "Genes", "gene", "gene_symbol", "SYMBOL", "X", "Unnamed: 0"])
    if gene_col is None:
        gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))


def parse_field(value, field):
    if pd.isna(value):
        return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None


def map_main_label(cls):
    if cls is None or pd.isna(cls):
        return None
    c = str(cls).strip().upper()
    if c.startswith("AML"):
        return "AML"
    if c == "CLL":
        return "CLL"
    if c == "CML":
        return "CML"
    if c == "MDS":
        return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c:
        return "HE"
    if "ALL" in c:
        return "ALL"
    return "OTHER"


def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None:
        X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2 / (2.0 * sigma ** 2)).astype(np.float32)


def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0:
        return 1.0
    sigma = float(np.mean(np.sqrt(upper)))
    if np.isfinite(sigma) and sigma > 0:
        return sigma
    return 1.0


def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    projections = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(projections + [bias])
    output = Dense(1, activation="sigmoid", use_bias=False,
                   kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                   name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model


def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model


def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]


def class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, weights)}


def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])


def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(specificity),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }


def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values) - 1)
    return (float(mean - tcrit * sem), float(mean + tcrit * sem))


def append_rows(path, rows):
    if not rows:
        return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


# ==============================================================================
# 3) INPUT CHECK
# ==============================================================================

print("\n[INPUT CHECK]")

for label, path in {
    "RA expression": RA_EXPR,
    "RA phenotype": RA_PHENO,
    "MILE expression": MILE_EXPR,
    "MILE phenotype": MILE_FULL_PHENO,
    "Pathway file": TOP_PATH_FILE
}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists():
        raise FileNotFoundError(path)


# ==============================================================================
# 4) LOAD RA
# ==============================================================================

print("\n[1/7] LOADING RA")

expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")

ra_sample_col = pick_col(ph_ra, ["sample", "Sample", "GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw", "group", "label"])

if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA", "HE"])].copy()

ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]

ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x == "RA" else 0 for x in ra_group.values], dtype=int)

print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full == 1))} | HE={int(np.sum(y_ra_full == 0))}")


# ==============================================================================
# 5) LOAD MILE / GSE13159
# ==============================================================================

print("\n[2/7] LOADING MILE / GSE13159")

expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")

mile_sample_col = pick_col(ph_mile, ["GSM", "sample", "Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])

if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()

available_samples = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(available_samples)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

cml_classes = ph_mile.loc[ph_mile["main_label"] == "CML", "leukemia_class"].value_counts()
print("\nCML leukemia_class breakdown:")
print(cml_classes)


# ==============================================================================
# 6) BUILD CML BONE-MARROW TARGET
# ==============================================================================

print("\n[3/7] BUILDING CML BONE-MARROW TARGET")

is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)

he_bm = ph_mile.loc[(ph_mile["main_label"] == "HE") & is_bm, mile_sample_col].tolist()
cml_bm = ph_mile.loc[(ph_mile["main_label"] == "CML") & is_bm, mile_sample_col].tolist()

print(f"CML bone marrow = {len(cml_bm)} | Healthy bone marrow = {len(he_bm)}")

if len(cml_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid CML/healthy bone-marrow target set found.")

target_samples_all = he_bm + cml_bm
y_target_all = np.array([0] * len(he_bm) + [1] * len(cml_bm), dtype=int)

print("Target total:", len(target_samples_all))
print("CML prevalence:", round(float(np.mean(y_target_all)), 4))


# ==============================================================================
# 7) COMMON GENES
# ==============================================================================

print("\n[4/7] COMMON GENES")

common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]

print("Common genes:", len(common_genes))


# ==============================================================================
# 8) LOAD TOP 116 ROBUST KEGG PATHWAYS
# ==============================================================================

print("\n[5/7] LOADING TOP 116 ROBUST KEGG PATHWAYS")

freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
pcol = pick_col(freq, ["Pathway"])
ccol = pick_col(freq, ["Selection_Count", "Count"])

if pcol is None or ccol is None:
    raise KeyError(f"Pathway-frequency columns not found: {list(freq.columns)}")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()

print("Downloading/loading KEGG_2021_Human...")
kegg = get_library(name=LIBRARY, organism="Human")

active_paths = []
for pathway in top_paths:
    if pathway in kegg:
        genes = sorted(set(kegg[pathway]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((pathway, genes))

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")

if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top116_Pathways.xlsx", index=False)


# ==============================================================================
# 8b) RESUME CHECK -- detect which repeats are already completed
# ==============================================================================

completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
    print(f"[RESUME] Will skip repeats: {sorted(completed_repeats)}")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")


# ==============================================================================
# 9) MAIN LOOP
# RA -> CML FREEZE & ADAPT
# ==============================================================================

print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON CML ({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):

    rep_number = rep + 1

    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    # --------------------------------------------------------------------
    # RA SOURCE SPLIT
    # --------------------------------------------------------------------

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)),
        test_size=RA_TEST_SIZE,
        stratify=y_ra_full,
        random_state=split_seed
    )

    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    # --------------------------------------------------------------------
    # RA DOMAIN STANDARDIZATION
    # --------------------------------------------------------------------

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes,
        columns=source_samples
    )

    # --------------------------------------------------------------------
    # RA SOURCE PATHWAY KERNELS
    # --------------------------------------------------------------------

    Ktr_list = []
    pathway_sigmas = {}

    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))

    Xtr = np.transpose(np.stack(Ktr_list), (1, 0, 2)).astype(np.float32)

    # --------------------------------------------------------------------
    # TRAIN SOURCE MODEL ON RA
    # --------------------------------------------------------------------

    clear_session()
    set_random_seed(split_seed)

    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(
        to_inputs(Xtr), y_source,
        epochs=EPOCHS_RA, batch_size=BATCH_RA, verbose=0,
        class_weight=class_weights(y_source), shuffle=True
    )
    source_weights = source_model.get_weights()

    # --------------------------------------------------------------------
    # CML TARGET STRATIFIED CV
    # --------------------------------------------------------------------

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):

        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        # scaler fitted ONLY on target TRAIN fold
        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list = []
        Ktest_list = []

        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1, 0, 2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1, 0, 2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed * 1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)

        target_model.fit(
            to_inputs(Xtrain_t), y_train_t,
            epochs=EPOCHS_ADAPT, batch_size=BATCH_ADAPT, verbose=0,
            class_weight=class_weights(y_train_t), shuffle=True
        )

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        metrics = compute_metrics(y_test_t, p_test, thr)
        metrics.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(metrics)

        for sample, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": sample,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session()
        gc.collect()

    # -------------------- CHECKPOINT: save this repeat's results now --------------------
    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session()
    gc.collect()


# ==============================================================================
# 10) FINAL TABLES -- reload full history from checkpoint (covers resumed runs)
# ==============================================================================

print("\n[7/7] FINAL SUMMARY")

metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[
    ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
     "Precision", "Recall", "Specificity", "F1"]
].mean()

summary_rows = []
for metric in ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
               "Precision", "Recall", "Specificity", "F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric,
        "N_repeats": len(values),
        "Mean": float(np.mean(values)),
        "SD": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        "Median": float(np.median(values)),
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
        "CI95_low": low,
        "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nPER-REPEAT RESULTS:")
print(repeat_means.round(4).to_string())

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))

print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")


# ==============================================================================
# 11) SAVE EVERYTHING
# ==============================================================================

out_xlsx = SAVE_PATH / f"RA2CML_MILE_FreezeAdapt_Top116_{REPEATS}reps.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat],
        "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    cml_classes.rename("N").reset_index().to_excel(writer, sheet_name="CML_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("=" * 110)
print("RA -> CML FREEZE & ADAPT COMPLETE")
print("=" * 110)

RA -> CML | MILE/GSE13159 | FREEZE & ADAPT | TOP 116 | LINUX GPU
Python executable : /home/altinbas-gpu/ra_leuk_project/venv/bin/python3
RA expression     : /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype      : /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression   : /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype    : /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file      : /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi.xlsx
Output directory  : /home/altinbas-gpu/ra_leuk_project/RA2CML_MILE_FREEZE_ADAPT_TOP116
REPEATS           : 80
N_SPLITS          : 2
EPOCHS_RA         : 400
EPOCHS_ADAPT      : 400
LR_ADAPT          : 0.0001
Visible GPUs      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_proje

  ... repeat 1/80 done | this repeat mean AUROC = 0.6689 | checkpoint saved


  ... repeat 2/80 done | this repeat mean AUROC = 0.7255 | checkpoint saved
  ... repeat 3/80 done | this repeat mean AUROC = 0.5986 | checkpoint saved
  ... repeat 4/80 done | this repeat mean AUROC = 0.7353 | checkpoint saved
  ... repeat 5/80 done | this repeat mean AUROC = 0.7215 | checkpoint saved
  ... repeat 6/80 done | this repeat mean AUROC = 0.8536 | checkpoint saved
  ... repeat 7/80 done | this repeat mean AUROC = 0.6426 | checkpoint saved
  ... repeat 8/80 done | this repeat mean AUROC = 0.6860 | checkpoint saved
  ... repeat 9/80 done | this repeat mean AUROC = 0.5981 | checkpoint saved
  ... repeat 10/80 done | this repeat mean AUROC = 0.6455 | checkpoint saved
  ... repeat 11/80 done | this repeat mean AUROC = 0.6374 | checkpoint saved
  ... repeat 12/80 done | this repeat mean AUROC = 0.6400 | checkpoint saved
  ... repeat 13/80 done | this repeat mean AUROC = 0.7544 | checkpoint saved
  ... repeat 14/80 done | this repeat mean AUROC = 0.5661 | checkpoint saved
  ... r

In [6]:
# ==============================================================================
# RA (SOURCE) -> MDS from MILE/GSE13159 (EXTERNAL TARGET, FREEZE & ADAPT)
# TOP 116 ROBUST KEGG PATHWAYS | BONE MARROW TISSUE-MATCHED
# LINUX GPU VERSION WITH CHECKPOINT/RESUME
#
# PROTOCOL:
#   - RA is the source domain
#   - MDS/HE bone marrow samples from MILE/GSE13159 form the target domain
#   - eta (pathway contribution) and bias are FROZEN from RA source training
#   - shared_projection (w) is ADAPTED on MDS target-TRAIN fold only
#   - MDS target-TEST fold remains completely held out
#   - Youden J threshold is calculated from target-TRAIN fold only
#   - Stratified CV is repeated across REPEATS
#
# CHECKPOINT: after every completed repeat, results are appended to a CSV.
# If interrupted (power loss, disconnect), re-running this script will skip
# already-completed repeats and resume from where it left off.
# ==============================================================================

import os
import sys
import gc
import random
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ==============================================================================
# 0) INSTALL REQUIRED PACKAGES IF MISSING
# ==============================================================================

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "tensorflow": "tensorflow",
    "gseapy": "gseapy"
}

for imp, pipn in required.items():
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pipn]
        )

import numpy as np
import pandas as pd
import scipy.stats as st

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

from gseapy import get_library


# ==============================================================================
# 1) SETTINGS -- Linux GPU makinesine göre ayarlandı, MDS için doğru hedef
# ==============================================================================

SEED = 42

BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"

MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"

TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi.xlsx"

# IMPORTANT: separate output folder from AML / CML runs
SAVE_PATH = BASE_DIR / "RA2MDS_MILE_FREEZE_ADAPT_TOP116"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Checkpoint files -- make the run resumable after interruption
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

LIBRARY = "KEGG_2021_Human"
TOP_N = 116

REPEATS = 80
N_SPLITS = 2          # AML/CML koşularında hızlı çalıştığı için aynı ayar kullanıldı
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20

# IMPORTANT: this is MDS
TARGET_DISEASE = "MDS"


# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

set_random_seed(SEED)
clear_session()


print("=" * 110)
print(
    f"RA -> {TARGET_DISEASE} | MILE/GSE13159 | "
    f"FREEZE & ADAPT | TOP {TOP_N} | LINUX GPU"
)
print("=" * 110)

print("Python executable :", sys.executable)
print("RA expression     :", RA_EXPR)
print("RA phenotype      :", RA_PHENO)
print("MILE expression   :", MILE_EXPR)
print("MILE phenotype    :", MILE_FULL_PHENO)
print("Pathway file      :", TOP_PATH_FILE)
print("Output directory  :", SAVE_PATH)
print("REPEATS           :", REPEATS)
print("N_SPLITS          :", N_SPLITS)
print("EPOCHS_RA         :", EPOCHS_RA)
print("EPOCHS_ADAPT      :", EPOCHS_ADAPT)
print("LR_ADAPT          :", LR_ADAPT)

print(
    "Visible GPUs      :",
    tf.config.list_physical_devices("GPU")
)


# ==============================================================================
# 2) HELPER FUNCTIONS
# ==============================================================================

def pick_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = str(c).strip().lower()
        if key in lookup:
            return lookup[key]
    return None


def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA", "NAN", "NONE", "---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)


def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name", "Gene", "Genes", "gene", "gene_symbol", "SYMBOL", "X", "Unnamed: 0"])
    if gene_col is None:
        gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))


def parse_field(value, field):
    if pd.isna(value):
        return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None


def map_main_label(cls):
    if cls is None or pd.isna(cls):
        return None
    c = str(cls).strip().upper()
    if c.startswith("AML"):
        return "AML"
    if c == "CLL":
        return "CLL"
    if c == "CML":
        return "CML"
    if c == "MDS":
        return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c:
        return "HE"
    if "ALL" in c:
        return "ALL"
    return "OTHER"


def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None:
        X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2 / (2.0 * sigma ** 2)).astype(np.float32)


def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0:
        return 1.0
    sigma = float(np.mean(np.sqrt(upper)))
    if np.isfinite(sigma) and sigma > 0:
        return sigma
    return 1.0


def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    projections = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(projections + [bias])
    output = Dense(1, activation="sigmoid", use_bias=False,
                   kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                   name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model


def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model


def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]


def class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, weights)}


def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])


def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(specificity),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }


def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values) - 1)
    return (float(mean - tcrit * sem), float(mean + tcrit * sem))


def append_rows(path, rows):
    if not rows:
        return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


# ==============================================================================
# 3) INPUT CHECK
# ==============================================================================

print("\n[INPUT CHECK]")

for label, path in {
    "RA expression": RA_EXPR,
    "RA phenotype": RA_PHENO,
    "MILE expression": MILE_EXPR,
    "MILE phenotype": MILE_FULL_PHENO,
    "Pathway file": TOP_PATH_FILE
}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists():
        raise FileNotFoundError(path)


# ==============================================================================
# 4) LOAD RA
# ==============================================================================

print("\n[1/7] LOADING RA")

expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")

ra_sample_col = pick_col(ph_ra, ["sample", "Sample", "GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw", "group", "label"])

if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA", "HE"])].copy()

ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]

ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x == "RA" else 0 for x in ra_group.values], dtype=int)

print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full == 1))} | HE={int(np.sum(y_ra_full == 0))}")


# ==============================================================================
# 5) LOAD MILE / GSE13159
# ==============================================================================

print("\n[2/7] LOADING MILE / GSE13159")

expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")

mile_sample_col = pick_col(ph_mile, ["GSM", "sample", "Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])

if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()

available_samples = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(available_samples)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

mds_classes = ph_mile.loc[ph_mile["main_label"] == "MDS", "leukemia_class"].value_counts()
print("\nMDS leukemia_class breakdown:")
print(mds_classes)


# ==============================================================================
# 6) BUILD MDS BONE-MARROW TARGET
# ==============================================================================

print("\n[3/7] BUILDING MDS BONE-MARROW TARGET")

is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)

he_bm = ph_mile.loc[(ph_mile["main_label"] == "HE") & is_bm, mile_sample_col].tolist()
mds_bm = ph_mile.loc[(ph_mile["main_label"] == "MDS") & is_bm, mile_sample_col].tolist()

print(f"MDS bone marrow = {len(mds_bm)} | Healthy bone marrow = {len(he_bm)}")

if len(mds_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid MDS/healthy bone-marrow target set found.")

target_samples_all = he_bm + mds_bm
y_target_all = np.array([0] * len(he_bm) + [1] * len(mds_bm), dtype=int)

print("Target total:", len(target_samples_all))
print("MDS prevalence:", round(float(np.mean(y_target_all)), 4))


# ==============================================================================
# 7) COMMON GENES
# ==============================================================================

print("\n[4/7] COMMON GENES")

common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]

print("Common genes:", len(common_genes))


# ==============================================================================
# 8) LOAD TOP 116 ROBUST KEGG PATHWAYS
# ==============================================================================

print("\n[5/7] LOADING TOP 116 ROBUST KEGG PATHWAYS")

freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
pcol = pick_col(freq, ["Pathway"])
ccol = pick_col(freq, ["Selection_Count", "Count"])

if pcol is None or ccol is None:
    raise KeyError(f"Pathway-frequency columns not found: {list(freq.columns)}")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()

print("Downloading/loading KEGG_2021_Human...")
kegg = get_library(name=LIBRARY, organism="Human")

active_paths = []
for pathway in top_paths:
    if pathway in kegg:
        genes = sorted(set(kegg[pathway]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((pathway, genes))

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")

if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top116_Pathways.xlsx", index=False)


# ==============================================================================
# 8b) RESUME CHECK -- detect which repeats are already completed
# ==============================================================================

completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
    print(f"[RESUME] Will skip repeats: {sorted(completed_repeats)}")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")


# ==============================================================================
# 9) MAIN LOOP
# RA -> MDS FREEZE & ADAPT
# ==============================================================================

print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON MDS ({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):

    rep_number = rep + 1

    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    # --------------------------------------------------------------------
    # RA SOURCE SPLIT
    # --------------------------------------------------------------------

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)),
        test_size=RA_TEST_SIZE,
        stratify=y_ra_full,
        random_state=split_seed
    )

    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    # --------------------------------------------------------------------
    # RA DOMAIN STANDARDIZATION
    # --------------------------------------------------------------------

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes,
        columns=source_samples
    )

    # --------------------------------------------------------------------
    # RA SOURCE PATHWAY KERNELS
    # --------------------------------------------------------------------

    Ktr_list = []
    pathway_sigmas = {}

    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))

    Xtr = np.transpose(np.stack(Ktr_list), (1, 0, 2)).astype(np.float32)

    # --------------------------------------------------------------------
    # TRAIN SOURCE MODEL ON RA
    # --------------------------------------------------------------------

    clear_session()
    set_random_seed(split_seed)

    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(
        to_inputs(Xtr), y_source,
        epochs=EPOCHS_RA, batch_size=BATCH_RA, verbose=0,
        class_weight=class_weights(y_source), shuffle=True
    )
    source_weights = source_model.get_weights()

    # --------------------------------------------------------------------
    # MDS TARGET STRATIFIED CV
    # --------------------------------------------------------------------

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):

        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        # scaler fitted ONLY on target TRAIN fold
        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list = []
        Ktest_list = []

        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1, 0, 2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1, 0, 2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed * 1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)

        target_model.fit(
            to_inputs(Xtrain_t), y_train_t,
            epochs=EPOCHS_ADAPT, batch_size=BATCH_ADAPT, verbose=0,
            class_weight=class_weights(y_train_t), shuffle=True
        )

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        metrics = compute_metrics(y_test_t, p_test, thr)
        metrics.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(metrics)

        for sample, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": sample,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session()
        gc.collect()

    # -------------------- CHECKPOINT: save this repeat's results now --------------------
    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session()
    gc.collect()


# ==============================================================================
# 10) FINAL TABLES -- reload full history from checkpoint (covers resumed runs)
# ==============================================================================

print("\n[7/7] FINAL SUMMARY")

metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[
    ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
     "Precision", "Recall", "Specificity", "F1"]
].mean()

summary_rows = []
for metric in ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
               "Precision", "Recall", "Specificity", "F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric,
        "N_repeats": len(values),
        "Mean": float(np.mean(values)),
        "SD": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        "Median": float(np.median(values)),
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
        "CI95_low": low,
        "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nPER-REPEAT RESULTS:")
print(repeat_means.round(4).to_string())

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))

print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")


# ==============================================================================
# 11) SAVE EVERYTHING
# ==============================================================================

out_xlsx = SAVE_PATH / f"RA2MDS_MILE_FreezeAdapt_Top116_{REPEATS}reps.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat],
        "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    mds_classes.rename("N").reset_index().to_excel(writer, sheet_name="MDS_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("=" * 110)
print("RA -> MDS FREEZE & ADAPT COMPLETE")
print("=" * 110)

RA -> MDS | MILE/GSE13159 | FREEZE & ADAPT | TOP 116 | LINUX GPU
Python executable : /home/altinbas-gpu/ra_leuk_project/venv/bin/python3
RA expression     : /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype      : /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression   : /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype    : /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file      : /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi.xlsx
Output directory  : /home/altinbas-gpu/ra_leuk_project/RA2MDS_MILE_FREEZE_ADAPT_TOP116
REPEATS           : 80
N_SPLITS          : 2
EPOCHS_RA         : 400
EPOCHS_ADAPT      : 400
LR_ADAPT          : 0.0001
Visible GPUs      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_proje

In [7]:
# ==============================================================================
# RA (SOURCE) -> ALL from MILE/GSE13159 (EXTERNAL TARGET, FREEZE & ADAPT)
# TOP 116 ROBUST KEGG PATHWAYS | BONE MARROW TISSUE-MATCHED
# LINUX GPU VERSION WITH CHECKPOINT/RESUME
#
# PROTOCOL:
#   - RA is the source domain
#   - ALL/HE bone marrow samples from MILE/GSE13159 form the target domain
#   - eta (pathway contribution) and bias are FROZEN from RA source training
#   - shared_projection (w) is ADAPTED on ALL target-TRAIN fold only
#   - ALL target-TEST fold remains completely held out
#   - Youden J threshold is calculated from target-TRAIN fold only
#   - Stratified CV is repeated across REPEATS
#
# CHECKPOINT: after every completed repeat, results are appended to a CSV.
# If interrupted (power loss, disconnect), re-running this script will skip
# already-completed repeats and resume from where it left off.
# ==============================================================================

import os
import sys
import gc
import random
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ==============================================================================
# 0) INSTALL REQUIRED PACKAGES IF MISSING
# ==============================================================================

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "tensorflow": "tensorflow",
    "gseapy": "gseapy"
}

for imp, pipn in required.items():
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pipn]
        )

import numpy as np
import pandas as pd
import scipy.stats as st

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

from gseapy import get_library


# ==============================================================================
# 1) SETTINGS -- Linux GPU makinesine göre ayarlandı, ALL için doğru hedef
# ==============================================================================

SEED = 42

BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"

MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"

TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi.xlsx"

# IMPORTANT: separate output folder from AML / CML / MDS runs
SAVE_PATH = BASE_DIR / "RA2ALL_MILE_FREEZE_ADAPT_TOP116"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Checkpoint files -- make the run resumable after interruption
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

LIBRARY = "KEGG_2021_Human"
TOP_N = 116

REPEATS = 80
N_SPLITS = 2          # AML/CML/MDS koşularında hızlı çalıştığı için aynı ayar kullanıldı
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20

# IMPORTANT: this is ALL (Acute Lymphoblastic Leukemia)
TARGET_DISEASE = "ALL"


# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

set_random_seed(SEED)
clear_session()


print("=" * 110)
print(
    f"RA -> {TARGET_DISEASE} | MILE/GSE13159 | "
    f"FREEZE & ADAPT | TOP {TOP_N} | LINUX GPU"
)
print("=" * 110)

print("Python executable :", sys.executable)
print("RA expression     :", RA_EXPR)
print("RA phenotype      :", RA_PHENO)
print("MILE expression   :", MILE_EXPR)
print("MILE phenotype    :", MILE_FULL_PHENO)
print("Pathway file      :", TOP_PATH_FILE)
print("Output directory  :", SAVE_PATH)
print("REPEATS           :", REPEATS)
print("N_SPLITS          :", N_SPLITS)
print("EPOCHS_RA         :", EPOCHS_RA)
print("EPOCHS_ADAPT      :", EPOCHS_ADAPT)
print("LR_ADAPT          :", LR_ADAPT)

print(
    "Visible GPUs      :",
    tf.config.list_physical_devices("GPU")
)


# ==============================================================================
# 2) HELPER FUNCTIONS
# ==============================================================================

def pick_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = str(c).strip().lower()
        if key in lookup:
            return lookup[key]
    return None


def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA", "NAN", "NONE", "---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)


def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name", "Gene", "Genes", "gene", "gene_symbol", "SYMBOL", "X", "Unnamed: 0"])
    if gene_col is None:
        gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))


def parse_field(value, field):
    if pd.isna(value):
        return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None


def map_main_label(cls):
    if cls is None or pd.isna(cls):
        return None
    c = str(cls).strip().upper()
    if c.startswith("AML"):
        return "AML"
    if c == "CLL":
        return "CLL"
    if c == "CML":
        return "CML"
    if c == "MDS":
        return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c:
        return "HE"
    if "ALL" in c:
        return "ALL"
    return "OTHER"


def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None:
        X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2 / (2.0 * sigma ** 2)).astype(np.float32)


def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0:
        return 1.0
    sigma = float(np.mean(np.sqrt(upper)))
    if np.isfinite(sigma) and sigma > 0:
        return sigma
    return 1.0


def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    projections = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(projections + [bias])
    output = Dense(1, activation="sigmoid", use_bias=False,
                   kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                   name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model


def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model


def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]


def class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, weights)}


def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])


def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(specificity),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }


def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values) - 1)
    return (float(mean - tcrit * sem), float(mean + tcrit * sem))


def append_rows(path, rows):
    if not rows:
        return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


# ==============================================================================
# 3) INPUT CHECK
# ==============================================================================

print("\n[INPUT CHECK]")

for label, path in {
    "RA expression": RA_EXPR,
    "RA phenotype": RA_PHENO,
    "MILE expression": MILE_EXPR,
    "MILE phenotype": MILE_FULL_PHENO,
    "Pathway file": TOP_PATH_FILE
}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists():
        raise FileNotFoundError(path)


# ==============================================================================
# 4) LOAD RA
# ==============================================================================

print("\n[1/7] LOADING RA")

expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")

ra_sample_col = pick_col(ph_ra, ["sample", "Sample", "GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw", "group", "label"])

if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA", "HE"])].copy()

ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]

ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x == "RA" else 0 for x in ra_group.values], dtype=int)

print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full == 1))} | HE={int(np.sum(y_ra_full == 0))}")


# ==============================================================================
# 5) LOAD MILE / GSE13159
# ==============================================================================

print("\n[2/7] LOADING MILE / GSE13159")

expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")

mile_sample_col = pick_col(ph_mile, ["GSM", "sample", "Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])

if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()

available_samples = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(available_samples)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

all_classes = ph_mile.loc[ph_mile["main_label"] == "ALL", "leukemia_class"].value_counts()
print("\nALL leukemia_class breakdown:")
print(all_classes)


# ==============================================================================
# 6) BUILD ALL BONE-MARROW TARGET
# ==============================================================================

print("\n[3/7] BUILDING ALL BONE-MARROW TARGET")

is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)

he_bm = ph_mile.loc[(ph_mile["main_label"] == "HE") & is_bm, mile_sample_col].tolist()
all_bm = ph_mile.loc[(ph_mile["main_label"] == "ALL") & is_bm, mile_sample_col].tolist()

print(f"ALL bone marrow = {len(all_bm)} | Healthy bone marrow = {len(he_bm)}")

if len(all_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid ALL/healthy bone-marrow target set found.")

target_samples_all = he_bm + all_bm
y_target_all = np.array([0] * len(he_bm) + [1] * len(all_bm), dtype=int)

print("Target total:", len(target_samples_all))
print("ALL prevalence:", round(float(np.mean(y_target_all)), 4))


# ==============================================================================
# 7) COMMON GENES
# ==============================================================================

print("\n[4/7] COMMON GENES")

common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]

print("Common genes:", len(common_genes))


# ==============================================================================
# 8) LOAD TOP 116 ROBUST KEGG PATHWAYS
# ==============================================================================

print("\n[5/7] LOADING TOP 116 ROBUST KEGG PATHWAYS")

freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
pcol = pick_col(freq, ["Pathway"])
ccol = pick_col(freq, ["Selection_Count", "Count"])

if pcol is None or ccol is None:
    raise KeyError(f"Pathway-frequency columns not found: {list(freq.columns)}")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()

print("Downloading/loading KEGG_2021_Human...")
kegg = get_library(name=LIBRARY, organism="Human")

active_paths = []
for pathway in top_paths:
    if pathway in kegg:
        genes = sorted(set(kegg[pathway]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((pathway, genes))

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")

if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top116_Pathways.xlsx", index=False)


# ==============================================================================
# 8b) RESUME CHECK -- detect which repeats are already completed
# ==============================================================================

completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
    print(f"[RESUME] Will skip repeats: {sorted(completed_repeats)}")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")


# ==============================================================================
# 9) MAIN LOOP
# RA -> ALL FREEZE & ADAPT
# ==============================================================================

print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON ALL ({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):

    rep_number = rep + 1

    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    # --------------------------------------------------------------------
    # RA SOURCE SPLIT
    # --------------------------------------------------------------------

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)),
        test_size=RA_TEST_SIZE,
        stratify=y_ra_full,
        random_state=split_seed
    )

    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    # --------------------------------------------------------------------
    # RA DOMAIN STANDARDIZATION
    # --------------------------------------------------------------------

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes,
        columns=source_samples
    )

    # --------------------------------------------------------------------
    # RA SOURCE PATHWAY KERNELS
    # --------------------------------------------------------------------

    Ktr_list = []
    pathway_sigmas = {}

    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))

    Xtr = np.transpose(np.stack(Ktr_list), (1, 0, 2)).astype(np.float32)

    # --------------------------------------------------------------------
    # TRAIN SOURCE MODEL ON RA
    # --------------------------------------------------------------------

    clear_session()
    set_random_seed(split_seed)

    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(
        to_inputs(Xtr), y_source,
        epochs=EPOCHS_RA, batch_size=BATCH_RA, verbose=0,
        class_weight=class_weights(y_source), shuffle=True
    )
    source_weights = source_model.get_weights()

    # --------------------------------------------------------------------
    # ALL TARGET STRATIFIED CV
    # --------------------------------------------------------------------

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):

        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        # scaler fitted ONLY on target TRAIN fold
        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list = []
        Ktest_list = []

        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1, 0, 2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1, 0, 2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed * 1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)

        target_model.fit(
            to_inputs(Xtrain_t), y_train_t,
            epochs=EPOCHS_ADAPT, batch_size=BATCH_ADAPT, verbose=0,
            class_weight=class_weights(y_train_t), shuffle=True
        )

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        metrics = compute_metrics(y_test_t, p_test, thr)
        metrics.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(metrics)

        for sample, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": sample,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session()
        gc.collect()

    # -------------------- CHECKPOINT: save this repeat's results now --------------------
    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session()
    gc.collect()


# ==============================================================================
# 10) FINAL TABLES -- reload full history from checkpoint (covers resumed runs)
# ==============================================================================

print("\n[7/7] FINAL SUMMARY")

metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[
    ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
     "Precision", "Recall", "Specificity", "F1"]
].mean()

summary_rows = []
for metric in ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
               "Precision", "Recall", "Specificity", "F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric,
        "N_repeats": len(values),
        "Mean": float(np.mean(values)),
        "SD": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        "Median": float(np.median(values)),
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
        "CI95_low": low,
        "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nPER-REPEAT RESULTS:")
print(repeat_means.round(4).to_string())

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))

print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")


# ==============================================================================
# 11) SAVE EVERYTHING
# ==============================================================================

out_xlsx = SAVE_PATH / f"RA2ALL_MILE_FreezeAdapt_Top116_{REPEATS}reps.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat],
        "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    all_classes.rename("N").reset_index().to_excel(writer, sheet_name="ALL_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("=" * 110)
print("RA -> ALL FREEZE & ADAPT COMPLETE")
print("=" * 110)

RA -> ALL | MILE/GSE13159 | FREEZE & ADAPT | TOP 116 | LINUX GPU
Python executable : /home/altinbas-gpu/ra_leuk_project/venv/bin/python3
RA expression     : /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype      : /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression   : /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype    : /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file      : /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi.xlsx
Output directory  : /home/altinbas-gpu/ra_leuk_project/RA2ALL_MILE_FREEZE_ADAPT_TOP116
REPEATS           : 80
N_SPLITS          : 2
EPOCHS_RA         : 400
EPOCHS_ADAPT      : 400
LR_ADAPT          : 0.0001
Visible GPUs      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_proje

I0000 00:00:1786423308.243356 2566430 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1786423311.450457 2566441 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 172 bytes spill stores, 172 bytes spill loads

I0000 00:00:1786423316.314137    8797 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 172 bytes spill stores, 172 bytes spill loads

I0000 00:00:1786423395.177099 2587524 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 336 bytes spill stores, 336 bytes spill loads

I0000 00:00:1786423399.055376    8803 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 336 bytes spill stores, 336 bytes spill load

  ... repeat 1/80 done | this repeat mean AUROC = 0.9289 | checkpoint saved


I0000 00:00:1786423547.852575    8780 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 172 bytes spill stores, 172 bytes spill loads

I0000 00:00:1786423627.036689    8785 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 336 bytes spill stores, 336 bytes spill loads



  ... repeat 2/80 done | this repeat mean AUROC = 0.9192 | checkpoint saved


...

  ... repeat 78/80 done | this repeat mean AUROC = 0.8685 | checkpoint saved


I0000 00:00:1786441208.446285    8781 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 172 bytes spill stores, 172 bytes spill loads

I0000 00:00:1786441288.638097    8788 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 336 bytes spill stores, 336 bytes spill loads



  ... repeat 79/80 done | this repeat mean AUROC = 0.9350 | checkpoint saved


I0000 00:00:1786441439.983222    8801 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 172 bytes spill stores, 172 bytes spill loads

I0000 00:00:1786441520.358601    8783 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'loop_add_fusion', 336 bytes spill stores, 336 bytes spill loads



  ... repeat 80/80 done | this repeat mean AUROC = 0.9455 | checkpoint saved

[7/7] FINAL SUMMARY

PER-REPEAT RESULTS:
         AUROC  PR_AUC  Accuracy  Balanced_Accuracy  Precision  Recall  Specificity      F1
Repeat                                                                                     
1       0.9289  0.9884    0.8709             0.8617     0.9831  0.8732       0.8502  0.9235
2       0.9192  0.9903    0.8289             0.8449     0.9834  0.8254       0.8645  0.8974
3       0.8966  0.9851    0.8135             0.8115     0.9765  0.8141       0.8089  0.8878
4       0.9366  0.9889    0.8953             0.8810     0.9845  0.8986       0.8634  0.9395
5       0.7057  0.9564    0.5286             0.6656     0.9603  0.4972       0.8341  0.6479
6       0.9415  0.9901    0.9183             0.9058     0.9879  0.9211       0.8904  0.9533
7       0.9049  0.9883    0.8455             0.8357     0.9794  0.8479       0.8236  0.9085
8       0.8086  0.9734    0.7140             0.7314  

In [9]:
# ==============================================================================
# RA (SOURCE) -> CLL from MILE/GSE13159 (EXTERNAL TARGET, FREEZE & ADAPT)
# TOP 116 ROBUST KEGG PATHWAYS | PERIPHERAL BLOOD TISSUE-MATCHED (see note below)
# LINUX GPU VERSION WITH CHECKPOINT/RESUME
#
# PROTOCOL:
#   - RA is the source domain
#   - CLL/HE peripheral blood samples from MILE/GSE13159 form the target domain
#     (CLL has no bone-marrow representation in this cohort -- see note in Step 6)
#   - eta (pathway contribution) and bias are FROZEN from RA source training
#   - shared_projection (w) is ADAPTED on CLL target-TRAIN fold only
#   - CLL target-TEST fold remains completely held out
#   - Youden J threshold is calculated from target-TRAIN fold only
#   - Stratified CV is repeated across REPEATS
#
# CHECKPOINT: after every completed repeat, results are appended to a CSV.
# If interrupted (power loss, disconnect), re-running this script will skip
# already-completed repeats and resume from where it left off.
# ==============================================================================

import os
import sys
import gc
import random
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ==============================================================================
# 0) INSTALL REQUIRED PACKAGES IF MISSING
# ==============================================================================

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "tensorflow": "tensorflow",
    "gseapy": "gseapy"
}

for imp, pipn in required.items():
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pipn]
        )

import numpy as np
import pandas as pd
import scipy.stats as st

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

from gseapy import get_library


# ==============================================================================
# 1) SETTINGS -- Linux GPU makinesine göre ayarlandı, CLL için doğru hedef
# ==============================================================================

SEED = 42

BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"

MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"

TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi.xlsx"

# IMPORTANT: separate output folder from AML / CML / MDS / ALL runs
SAVE_PATH = BASE_DIR / "RA2CLL_MILE_FREEZE_ADAPT_TOP116"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Checkpoint files -- make the run resumable after interruption
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

LIBRARY = "KEGG_2021_Human"
TOP_N = 116

REPEATS = 80
N_SPLITS = 2          # AML/CML/MDS/ALL koşularında hızlı çalıştığı için aynı ayar kullanıldı
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20

# IMPORTANT: this is CLL (Chronic Lymphocytic Leukemia)
TARGET_DISEASE = "CLL"


# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

set_random_seed(SEED)
clear_session()


print("=" * 110)
print(
    f"RA -> {TARGET_DISEASE} | MILE/GSE13159 | "
    f"FREEZE & ADAPT | TOP {TOP_N} | LINUX GPU"
)
print("=" * 110)

print("Python executable :", sys.executable)
print("RA expression     :", RA_EXPR)
print("RA phenotype      :", RA_PHENO)
print("MILE expression   :", MILE_EXPR)
print("MILE phenotype    :", MILE_FULL_PHENO)
print("Pathway file      :", TOP_PATH_FILE)
print("Output directory  :", SAVE_PATH)
print("REPEATS           :", REPEATS)
print("N_SPLITS          :", N_SPLITS)
print("EPOCHS_RA         :", EPOCHS_RA)
print("EPOCHS_ADAPT      :", EPOCHS_ADAPT)
print("LR_ADAPT          :", LR_ADAPT)

print(
    "Visible GPUs      :",
    tf.config.list_physical_devices("GPU")
)


# ==============================================================================
# 2) HELPER FUNCTIONS
# ==============================================================================

def pick_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = str(c).strip().lower()
        if key in lookup:
            return lookup[key]
    return None


def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA", "NAN", "NONE", "---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)


def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name", "Gene", "Genes", "gene", "gene_symbol", "SYMBOL", "X", "Unnamed: 0"])
    if gene_col is None:
        gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))


def parse_field(value, field):
    if pd.isna(value):
        return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None


def map_main_label(cls):
    if cls is None or pd.isna(cls):
        return None
    c = str(cls).strip().upper()
    if c.startswith("AML"):
        return "AML"
    if c == "CLL":
        return "CLL"
    if c == "CML":
        return "CML"
    if c == "MDS":
        return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c:
        return "HE"
    if "ALL" in c:
        return "ALL"
    return "OTHER"


def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None:
        X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2 / (2.0 * sigma ** 2)).astype(np.float32)


def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0:
        return 1.0
    sigma = float(np.mean(np.sqrt(upper)))
    if np.isfinite(sigma) and sigma > 0:
        return sigma
    return 1.0


def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    projections = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(projections + [bias])
    output = Dense(1, activation="sigmoid", use_bias=False,
                   kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                   name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model


def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model


def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]


def class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, weights)}


def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])


def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(specificity),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }


def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values) - 1)
    return (float(mean - tcrit * sem), float(mean + tcrit * sem))


def append_rows(path, rows):
    if not rows:
        return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


# ==============================================================================
# 3) INPUT CHECK
# ==============================================================================

print("\n[INPUT CHECK]")

for label, path in {
    "RA expression": RA_EXPR,
    "RA phenotype": RA_PHENO,
    "MILE expression": MILE_EXPR,
    "MILE phenotype": MILE_FULL_PHENO,
    "Pathway file": TOP_PATH_FILE
}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists():
        raise FileNotFoundError(path)


# ==============================================================================
# 4) LOAD RA
# ==============================================================================

print("\n[1/7] LOADING RA")

expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")

ra_sample_col = pick_col(ph_ra, ["sample", "Sample", "GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw", "group", "label"])

if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA", "HE"])].copy()

ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]

ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x == "RA" else 0 for x in ra_group.values], dtype=int)

print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full == 1))} | HE={int(np.sum(y_ra_full == 0))}")


# ==============================================================================
# 5) LOAD MILE / GSE13159
# ==============================================================================

print("\n[2/7] LOADING MILE / GSE13159")

expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")

mile_sample_col = pick_col(ph_mile, ["GSM", "sample", "Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])

if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()

available_samples = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(available_samples)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

cll_classes = ph_mile.loc[ph_mile["main_label"] == "CLL", "leukemia_class"].value_counts()
print("\nCLL leukemia_class breakdown:")
print(cll_classes)


# ==============================================================================
# 6) BUILD CLL TARGET
# ==============================================================================
#
# IMPORTANT TISSUE NOTE:
# Unlike AML/CML/MDS/ALL, MILE (GSE13159) contains ZERO bone-marrow CLL
# samples -- all 448 CLL cases in this cohort are peripheral blood.
# CLL is clinically characterized by circulating malignant lymphocytes in
# peripheral blood, which is consistent with this collection pattern.
# To keep the CLL arm tissue-matched (case vs. control from the SAME
# specimen type), healthy controls are also drawn from peripheral blood
# here, rather than bone marrow as in the other four disease arms.
# This deviates from the bone-marrow matching used for AML/CML/MDS/ALL and
# should be reported explicitly as such in any manuscript text.
# ==============================================================================

print("\n[3/7] BUILDING CLL TARGET (PERIPHERAL BLOOD -- see tissue note above)")

is_pb = ph_mile["sample_type_norm"].str.contains("peripheral blood", na=False)

he_pb = ph_mile.loc[(ph_mile["main_label"] == "HE") & is_pb, mile_sample_col].tolist()
cll_pb = ph_mile.loc[(ph_mile["main_label"] == "CLL") & is_pb, mile_sample_col].tolist()

print(f"CLL peripheral blood = {len(cll_pb)} | Healthy peripheral blood = {len(he_pb)}")

if len(cll_pb) == 0 or len(he_pb) == 0:
    # Fallback: if there are no HE peripheral-blood controls either,
    # use all available HE samples regardless of tissue (last resort).
    print("WARNING: insufficient peripheral-blood HE controls; "
          "falling back to ALL available HE samples regardless of tissue.")
    he_pb = ph_mile.loc[ph_mile["main_label"] == "HE", mile_sample_col].tolist()
    print(f"Fallback HE pool size: {len(he_pb)}")

if len(cll_pb) == 0 or len(he_pb) == 0:
    raise ValueError("No valid CLL/healthy target set found even after fallback.")

target_samples_all = he_pb + cll_pb
y_target_all = np.array([0] * len(he_pb) + [1] * len(cll_pb), dtype=int)

print("Target total:", len(target_samples_all))
print("CLL prevalence:", round(float(np.mean(y_target_all)), 4))


# ==============================================================================
# 7) COMMON GENES
# ==============================================================================

print("\n[4/7] COMMON GENES")

common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]

print("Common genes:", len(common_genes))


# ==============================================================================
# 8) LOAD TOP 116 ROBUST KEGG PATHWAYS
# ==============================================================================

print("\n[5/7] LOADING TOP 116 ROBUST KEGG PATHWAYS")

freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
pcol = pick_col(freq, ["Pathway"])
ccol = pick_col(freq, ["Selection_Count", "Count"])

if pcol is None or ccol is None:
    raise KeyError(f"Pathway-frequency columns not found: {list(freq.columns)}")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()

print("Downloading/loading KEGG_2021_Human...")
kegg = get_library(name=LIBRARY, organism="Human")

active_paths = []
for pathway in top_paths:
    if pathway in kegg:
        genes = sorted(set(kegg[pathway]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((pathway, genes))

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")

if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top116_Pathways.xlsx", index=False)


# ==============================================================================
# 8b) RESUME CHECK -- detect which repeats are already completed
# ==============================================================================

completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
    print(f"[RESUME] Will skip repeats: {sorted(completed_repeats)}")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")


# ==============================================================================
# 9) MAIN LOOP
# RA -> CLL FREEZE & ADAPT
# ==============================================================================

print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON CLL ({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):

    rep_number = rep + 1

    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    # --------------------------------------------------------------------
    # RA SOURCE SPLIT
    # --------------------------------------------------------------------

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)),
        test_size=RA_TEST_SIZE,
        stratify=y_ra_full,
        random_state=split_seed
    )

    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    # --------------------------------------------------------------------
    # RA DOMAIN STANDARDIZATION
    # --------------------------------------------------------------------

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes,
        columns=source_samples
    )

    # --------------------------------------------------------------------
    # RA SOURCE PATHWAY KERNELS
    # --------------------------------------------------------------------

    Ktr_list = []
    pathway_sigmas = {}

    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))

    Xtr = np.transpose(np.stack(Ktr_list), (1, 0, 2)).astype(np.float32)

    # --------------------------------------------------------------------
    # TRAIN SOURCE MODEL ON RA
    # --------------------------------------------------------------------

    clear_session()
    set_random_seed(split_seed)

    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(
        to_inputs(Xtr), y_source,
        epochs=EPOCHS_RA, batch_size=BATCH_RA, verbose=0,
        class_weight=class_weights(y_source), shuffle=True
    )
    source_weights = source_model.get_weights()

    # --------------------------------------------------------------------
    # CLL TARGET STRATIFIED CV
    # --------------------------------------------------------------------

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):

        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        # scaler fitted ONLY on target TRAIN fold
        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list = []
        Ktest_list = []

        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1, 0, 2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1, 0, 2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed * 1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)

        target_model.fit(
            to_inputs(Xtrain_t), y_train_t,
            epochs=EPOCHS_ADAPT, batch_size=BATCH_ADAPT, verbose=0,
            class_weight=class_weights(y_train_t), shuffle=True
        )

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        metrics = compute_metrics(y_test_t, p_test, thr)
        metrics.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(metrics)

        for sample, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": sample,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session()
        gc.collect()

    # -------------------- CHECKPOINT: save this repeat's results now --------------------
    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session()
    gc.collect()


# ==============================================================================
# 10) FINAL TABLES -- reload full history from checkpoint (covers resumed runs)
# ==============================================================================

print("\n[7/7] FINAL SUMMARY")

metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[
    ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
     "Precision", "Recall", "Specificity", "F1"]
].mean()

summary_rows = []
for metric in ["AUROC", "PR_AUC", "Accuracy", "Balanced_Accuracy",
               "Precision", "Recall", "Specificity", "F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric,
        "N_repeats": len(values),
        "Mean": float(np.mean(values)),
        "SD": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        "Median": float(np.median(values)),
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
        "CI95_low": low,
        "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nPER-REPEAT RESULTS:")
print(repeat_means.round(4).to_string())

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))

print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")


# ==============================================================================
# 11) SAVE EVERYTHING
# ==============================================================================

out_xlsx = SAVE_PATH / f"RA2CLL_MILE_FreezeAdapt_Top116_{REPEATS}reps.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat],
        "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    cll_classes.rename("N").reset_index().to_excel(writer, sheet_name="CLL_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("=" * 110)
print("RA -> CLL FREEZE & ADAPT COMPLETE")
print("=" * 110)

RA -> CLL | MILE/GSE13159 | FREEZE & ADAPT | TOP 116 | LINUX GPU
Python executable : /home/altinbas-gpu/ra_leuk_project/venv/bin/python3
RA expression     : /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype      : /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression   : /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype    : /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file      : /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi.xlsx
Output directory  : /home/altinbas-gpu/ra_leuk_project/RA2CLL_MILE_FREEZE_ADAPT_TOP116
REPEATS           : 80
N_SPLITS          : 2
EPOCHS_RA         : 400
EPOCHS_ADAPT      : 400
LR_ADAPT          : 0.0001
Visible GPUs      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_proje

In [2]:
# ==============================================================================
# RA (SOURCE) -> AML from MILE/GSE13159 | PID (NCI-Nature_2016) TOP-105
# LINUX GPU | SINGLE BLOCK | CHECKPOINT/RESUME
# ==============================================================================

import os, sys, gc, random, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

required = {"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn",
            "openpyxl":"openpyxl","tensorflow":"tensorflow","gseapy":"gseapy"}
for imp, pipn in required.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pipn])

import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed
from gseapy import get_library

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------
SEED = 42
BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"
MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"
TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi_NCI-Nature_2016.xlsx"

LIBRARY = "NCI-Nature_2016"
TOP_N = 105

SAVE_PATH = BASE_DIR / "RA2AML_PID_FREEZE_ADAPT_TOP105"
SAVE_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

REPEATS = 80
N_SPLITS = 2
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20
TARGET_DISEASE = "AML"

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
set_random_seed(SEED); clear_session()

print("="*100)
print(f"RA -> {TARGET_DISEASE} (MILE) | PID (NCI-Nature_2016) | TOP {TOP_N} | REPEATS={REPEATS}")
print("="*100)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def pick_col(df, cands):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in cands:
        k = str(c).strip().lower()
        if k in lookup: return lookup[k]
    return None

def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA","NAN","NONE","---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)

def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name","Gene","Genes","gene","gene_symbol","SYMBOL","X","Unnamed: 0"])
    if gene_col is None: gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))

def parse_field(value, field):
    if pd.isna(value): return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None

def map_main_label(cls):
    if cls is None or pd.isna(cls): return None
    c = str(cls).strip().upper()
    if c.startswith("AML"): return "AML"
    if c == "CLL": return "CLL"
    if c == "CML": return "CML"
    if c == "MDS": return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c: return "HE"
    if "ALL" in c: return "ALL"
    return "OTHER"

def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None: X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2/(2.0*sigma**2)).astype(np.float32)

def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0: return 1.0
    s = float(np.mean(np.sqrt(upper)))
    return s if np.isfinite(s) and s > 0 else 1.0

def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    proj = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(proj + [bias])
    out = Dense(1, activation="sigmoid", use_bias=False,
                kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model

def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model

def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]

def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, w)}

def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(spec),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }

def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2: return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values)-1)
    return (float(mean - tcrit*sem), float(mean + tcrit*sem))

def append_rows(path, rows):
    if not rows: return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)

# ------------------------------------------------------------------
# INPUT CHECK
# ------------------------------------------------------------------
print("\n[INPUT CHECK]")
for label, path in {"RA expression":RA_EXPR, "RA phenotype":RA_PHENO,
                     "MILE expression":MILE_EXPR, "MILE phenotype":MILE_FULL_PHENO,
                     "Pathway file (PID)":TOP_PATH_FILE}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists(): raise FileNotFoundError(path)

# ------------------------------------------------------------------
# LOAD RA
# ------------------------------------------------------------------
print("\n[1/7] LOADING RA")
expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")
ra_sample_col = pick_col(ph_ra, ["sample","Sample","GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw","group","label"])
if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA","HE"])].copy()
ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]
ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x=="RA" else 0 for x in ra_group.values], dtype=int)
print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full==1))} | HE={int(np.sum(y_ra_full==0))}")

# ------------------------------------------------------------------
# LOAD MILE
# ------------------------------------------------------------------
print("\n[2/7] LOADING MILE / GSE13159")
expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")
mile_sample_col = pick_col(ph_mile, ["GSM","sample","Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])
if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()
avail = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(avail)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

aml_subtypes = ph_mile.loc[ph_mile["main_label"]=="AML", "leukemia_class"].value_counts()
print("\nAML leukemia_class breakdown:")
print(aml_subtypes)

# ------------------------------------------------------------------
# BUILD AML BONE-MARROW TARGET
# ------------------------------------------------------------------
print(f"\n[3/7] BUILDING {TARGET_DISEASE} BONE-MARROW TARGET")
is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)
he_bm = ph_mile.loc[(ph_mile["main_label"]=="HE") & is_bm, mile_sample_col].tolist()
case_bm = ph_mile.loc[(ph_mile["main_label"]==TARGET_DISEASE) & is_bm, mile_sample_col].tolist()
print(f"{TARGET_DISEASE} bone marrow = {len(case_bm)} | Healthy bone marrow = {len(he_bm)}")
if len(case_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid AML/healthy bone-marrow target set found.")

target_samples_all = he_bm + case_bm
y_target_all = np.array([0]*len(he_bm) + [1]*len(case_bm), dtype=int)
print("Target total:", len(target_samples_all), "| AML prevalence:", round(float(np.mean(y_target_all)),4))

# ------------------------------------------------------------------
# COMMON GENES
# ------------------------------------------------------------------
print("\n[4/7] COMMON GENES")
common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]
print("Common genes:", len(common_genes))

# ------------------------------------------------------------------
# LOAD TOP-105 PID PATHWAYS
# ------------------------------------------------------------------
print("\n[5/7] LOADING TOP-105 ROBUST PID (NCI-Nature_2016) PATHWAYS")
freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
print("Pathway file columns found:", list(freq.columns))
pcol = pick_col(freq, ["Pathway","Pathway_Name","Name"])
ccol = pick_col(freq, ["Selection_Count","Count","Frequency","Selection_Frequency"])
if pcol is None or ccol is None:
    raise KeyError(f"Could not detect columns. Columns present: {list(freq.columns)}")
print(f"Using pathway column = '{pcol}', count column = '{ccol}'")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()
print(f"Downloading/loading {LIBRARY}...")
pid = get_library(name=LIBRARY, organism="Human")

active_paths = []
unmatched = []
for p in top_paths:
    if p in pid:
        genes = sorted(set(pid[p]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((p, genes))
    else:
        unmatched.append(p)

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")
if unmatched:
    print(f"WARNING: {len(unmatched)} unmatched. First few: {unmatched[:5]}")
if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top105_PID_Pathways.xlsx", index=False)

# ------------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------------
completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")

# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON {TARGET_DISEASE} "
      f"({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):
    rep_number = rep + 1
    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)), test_size=RA_TEST_SIZE,
        stratify=y_ra_full, random_state=split_seed
    )
    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes, columns=source_samples
    )

    Ktr_list, pathway_sigmas = [], {}
    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))
    Xtr = np.transpose(np.stack(Ktr_list), (1,0,2)).astype(np.float32)

    clear_session(); set_random_seed(split_seed)
    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(to_inputs(Xtr), y_source, epochs=EPOCHS_RA, batch_size=BATCH_RA,
                      verbose=0, class_weight=class_weights(y_source), shuffle=True)
    source_weights = source_model.get_weights()

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):
        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list, Ktest_list = [], []
        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1,0,2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1,0,2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed*1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)
        target_model.fit(to_inputs(Xtrain_t), y_train_t, epochs=EPOCHS_ADAPT,
                          batch_size=BATCH_ADAPT, verbose=0,
                          class_weight=class_weights(y_train_t), shuffle=True)

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        met = compute_metrics(y_test_t, p_test, thr)
        met.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(met)

        for s, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": s,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session(); gc.collect()

    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session(); gc.collect()

# ------------------------------------------------------------------
# SUMMARY + SAVE
# ------------------------------------------------------------------
print("\n[7/7] SUMMARY")
metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[["AUROC","PR_AUC","Accuracy","Balanced_Accuracy",
                                              "Precision","Recall","Specificity","F1"]].mean()

summary_rows = []
for metric in ["AUROC","PR_AUC","Accuracy","Balanced_Accuracy","Precision","Recall","Specificity","F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric, "N_repeats": len(values),
        "Mean": float(np.mean(values)), "SD": float(np.std(values, ddof=1)) if len(values)>1 else np.nan,
        "Median": float(np.median(values)), "Min": float(np.min(values)), "Max": float(np.max(values)),
        "CI95_low": low, "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))
print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")

out_xlsx = SAVE_PATH / f"RA2{TARGET_DISEASE}_PID_FreezeAdapt_Top105_{REPEATS}reps.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat], "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    aml_subtypes.reset_index().to_excel(writer, sheet_name="AML_Subtype_Check", index=False)

print("\nSaved:", out_xlsx)
print("="*100)
print("DONE.")
print("="*100)

I0000 00:00:1786518681.499448    7758 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786518683.012511    7758 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


RA -> AML (MILE) | PID (NCI-Nature_2016) | TOP 105 | REPEATS=80
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression     : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype      : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file (PID)  : True | /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi_NCI-Nature_2016.xlsx

[1/7] LOADING RA
Reading combat_corrected_by_gse.xlsx | gene column=Gen_name
RA n=195 | RA=163 | HE=32

[2/7] LOADING MILE / GSE13159
Reading GSE13159_gene_unique.xlsx | gene column=gene

MILE label x sample type:
sample_type_norm  bone marrow  peripheral blood
main_label                                     
ALL                       710                40
AML                 

In [3]:
# ==============================================================================
# RA (SOURCE) -> CML from MILE/GSE13159 | PID (NCI-Nature_2016) TOP-105
# LINUX GPU | SINGLE BLOCK | CHECKPOINT/RESUME
# ==============================================================================

import os, sys, gc, random, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

required = {"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn",
            "openpyxl":"openpyxl","tensorflow":"tensorflow","gseapy":"gseapy"}
for imp, pipn in required.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pipn])

import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed
from gseapy import get_library

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------
SEED = 42
BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"
MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"
TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi_NCI-Nature_2016.xlsx"

LIBRARY = "NCI-Nature_2016"
TOP_N = 105

SAVE_PATH = BASE_DIR / "RA2CML_PID_FREEZE_ADAPT_TOP105"
SAVE_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

REPEATS = 80
N_SPLITS = 2
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20
TARGET_DISEASE = "CML"

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
set_random_seed(SEED); clear_session()

print("="*100)
print(f"RA -> {TARGET_DISEASE} (MILE) | PID (NCI-Nature_2016) | TOP {TOP_N} | REPEATS={REPEATS}")
print("="*100)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def pick_col(df, cands):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in cands:
        k = str(c).strip().lower()
        if k in lookup: return lookup[k]
    return None

def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA","NAN","NONE","---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)

def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name","Gene","Genes","gene","gene_symbol","SYMBOL","X","Unnamed: 0"])
    if gene_col is None: gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))

def parse_field(value, field):
    if pd.isna(value): return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None

def map_main_label(cls):
    if cls is None or pd.isna(cls): return None
    c = str(cls).strip().upper()
    if c.startswith("AML"): return "AML"
    if c == "CLL": return "CLL"
    if c == "CML": return "CML"
    if c == "MDS": return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c: return "HE"
    if "ALL" in c: return "ALL"
    return "OTHER"

def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None: X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2/(2.0*sigma**2)).astype(np.float32)

def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0: return 1.0
    s = float(np.mean(np.sqrt(upper)))
    return s if np.isfinite(s) and s > 0 else 1.0

def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    proj = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(proj + [bias])
    out = Dense(1, activation="sigmoid", use_bias=False,
                kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model

def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model

def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]

def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, w)}

def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(spec),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }

def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2: return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values)-1)
    return (float(mean - tcrit*sem), float(mean + tcrit*sem))

def append_rows(path, rows):
    if not rows: return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)

# ------------------------------------------------------------------
# INPUT CHECK
# ------------------------------------------------------------------
print("\n[INPUT CHECK]")
for label, path in {"RA expression":RA_EXPR, "RA phenotype":RA_PHENO,
                     "MILE expression":MILE_EXPR, "MILE phenotype":MILE_FULL_PHENO,
                     "Pathway file (PID)":TOP_PATH_FILE}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists(): raise FileNotFoundError(path)

# ------------------------------------------------------------------
# LOAD RA
# ------------------------------------------------------------------
print("\n[1/7] LOADING RA")
expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")
ra_sample_col = pick_col(ph_ra, ["sample","Sample","GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw","group","label"])
if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA","HE"])].copy()
ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]
ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x=="RA" else 0 for x in ra_group.values], dtype=int)
print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full==1))} | HE={int(np.sum(y_ra_full==0))}")

# ------------------------------------------------------------------
# LOAD MILE
# ------------------------------------------------------------------
print("\n[2/7] LOADING MILE / GSE13159")
expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")
mile_sample_col = pick_col(ph_mile, ["GSM","sample","Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])
if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()
avail = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(avail)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

cml_classes = ph_mile.loc[ph_mile["main_label"]=="CML", "leukemia_class"].value_counts()
print("\nCML leukemia_class breakdown:")
print(cml_classes)

# ------------------------------------------------------------------
# BUILD CML BONE-MARROW TARGET
# ------------------------------------------------------------------
print(f"\n[3/7] BUILDING {TARGET_DISEASE} BONE-MARROW TARGET")
is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)
he_bm = ph_mile.loc[(ph_mile["main_label"]=="HE") & is_bm, mile_sample_col].tolist()
case_bm = ph_mile.loc[(ph_mile["main_label"]==TARGET_DISEASE) & is_bm, mile_sample_col].tolist()
print(f"{TARGET_DISEASE} bone marrow = {len(case_bm)} | Healthy bone marrow = {len(he_bm)}")
if len(case_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid CML/healthy bone-marrow target set found.")

target_samples_all = he_bm + case_bm
y_target_all = np.array([0]*len(he_bm) + [1]*len(case_bm), dtype=int)
print("Target total:", len(target_samples_all), "| CML prevalence:", round(float(np.mean(y_target_all)),4))

# ------------------------------------------------------------------
# COMMON GENES
# ------------------------------------------------------------------
print("\n[4/7] COMMON GENES")
common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]
print("Common genes:", len(common_genes))

# ------------------------------------------------------------------
# LOAD TOP-105 PID PATHWAYS
# ------------------------------------------------------------------
print("\n[5/7] LOADING TOP-105 ROBUST PID (NCI-Nature_2016) PATHWAYS")
freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
print("Pathway file columns found:", list(freq.columns))
pcol = pick_col(freq, ["Pathway","Pathway_Name","Name"])
ccol = pick_col(freq, ["Selection_Count","Count","Frequency","Selection_Frequency"])
if pcol is None or ccol is None:
    raise KeyError(f"Could not detect columns. Columns present: {list(freq.columns)}")
print(f"Using pathway column = '{pcol}', count column = '{ccol}'")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()
print(f"Downloading/loading {LIBRARY}...")
pid = get_library(name=LIBRARY, organism="Human")

active_paths = []
unmatched = []
for p in top_paths:
    if p in pid:
        genes = sorted(set(pid[p]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((p, genes))
    else:
        unmatched.append(p)

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")
if unmatched:
    print(f"WARNING: {len(unmatched)} unmatched. First few: {unmatched[:5]}")
if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top105_PID_Pathways.xlsx", index=False)

# ------------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------------
completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")

# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON {TARGET_DISEASE} "
      f"({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):
    rep_number = rep + 1
    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)), test_size=RA_TEST_SIZE,
        stratify=y_ra_full, random_state=split_seed
    )
    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes, columns=source_samples
    )

    Ktr_list, pathway_sigmas = [], {}
    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))
    Xtr = np.transpose(np.stack(Ktr_list), (1,0,2)).astype(np.float32)

    clear_session(); set_random_seed(split_seed)
    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(to_inputs(Xtr), y_source, epochs=EPOCHS_RA, batch_size=BATCH_RA,
                      verbose=0, class_weight=class_weights(y_source), shuffle=True)
    source_weights = source_model.get_weights()

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):
        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list, Ktest_list = [], []
        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1,0,2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1,0,2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed*1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)
        target_model.fit(to_inputs(Xtrain_t), y_train_t, epochs=EPOCHS_ADAPT,
                          batch_size=BATCH_ADAPT, verbose=0,
                          class_weight=class_weights(y_train_t), shuffle=True)

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        met = compute_metrics(y_test_t, p_test, thr)
        met.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(met)

        for s, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": s,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session(); gc.collect()

    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session(); gc.collect()

# ------------------------------------------------------------------
# SUMMARY + SAVE
# ------------------------------------------------------------------
print("\n[7/7] SUMMARY")
metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[["AUROC","PR_AUC","Accuracy","Balanced_Accuracy",
                                              "Precision","Recall","Specificity","F1"]].mean()

summary_rows = []
for metric in ["AUROC","PR_AUC","Accuracy","Balanced_Accuracy","Precision","Recall","Specificity","F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric, "N_repeats": len(values),
        "Mean": float(np.mean(values)), "SD": float(np.std(values, ddof=1)) if len(values)>1 else np.nan,
        "Median": float(np.median(values)), "Min": float(np.min(values)), "Max": float(np.max(values)),
        "CI95_low": low, "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))
print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")

out_xlsx = SAVE_PATH / f"RA2{TARGET_DISEASE}_PID_FreezeAdapt_Top105_{REPEATS}reps.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat], "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    cml_classes.reset_index().to_excel(writer, sheet_name="CML_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("="*100)
print("DONE.")
print("="*100)

RA -> CML (MILE) | PID (NCI-Nature_2016) | TOP 105 | REPEATS=80
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression     : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype      : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file (PID)  : True | /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi_NCI-Nature_2016.xlsx

[1/7] LOADING RA
Reading combat_corrected_by_gse.xlsx | gene column=Gen_name
RA n=195 | RA=163 | HE=32

[2/7] LOADING MILE / GSE13159
Reading GSE13159_gene_unique.xlsx | gene column=gene

MILE label x sample type:
sample_type_norm  bone marrow  peripheral blood
main_label                                     
ALL                       710                40
AML                 

  ... repeat 9/80 done | this repeat mean AUROC = 0.6580 | checkpoint saved


  ... repeat 10/80 done | this repeat mean AUROC = 0.7727 | checkpoint saved
  ... repeat 11/80 done | this repeat mean AUROC = 0.6095 | checkpoint saved
  ... repeat 12/80 done | this repeat mean AUROC = 0.6662 | checkpoint saved
  ... repeat 13/80 done | this repeat mean AUROC = 0.7744 | checkpoint saved
  ... repeat 14/80 done | this repeat mean AUROC = 0.6827 | checkpoint saved
  ... repeat 15/80 done | this repeat mean AUROC = 0.7336 | checkpoint saved
  ... repeat 16/80 done | this repeat mean AUROC = 0.7599 | checkpoint saved
  ... repeat 17/80 done | this repeat mean AUROC = 0.7730 | checkpoint saved
  ... repeat 18/80 done | this repeat mean AUROC = 0.6732 | checkpoint saved
  ... repeat 19/80 done | this repeat mean AUROC = 0.6759 | checkpoint saved
  ... repeat 20/80 done | this repeat mean AUROC = 0.6280 | checkpoint saved
  ... repeat 21/80 done | this repeat mean AUROC = 0.6584 | checkpoint saved
  ... repeat 22/80 done | this repeat mean AUROC = 0.5707 | checkpoint saved

In [4]:
# ==============================================================================
# RA (SOURCE) -> MDS from MILE/GSE13159 | PID (NCI-Nature_2016) TOP-105
# LINUX GPU | SINGLE BLOCK | CHECKPOINT/RESUME
# ==============================================================================

import os, sys, gc, random, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

required = {"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn",
            "openpyxl":"openpyxl","tensorflow":"tensorflow","gseapy":"gseapy"}
for imp, pipn in required.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pipn])

import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed
from gseapy import get_library

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------
SEED = 42
BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"
MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"
TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi_NCI-Nature_2016.xlsx"

LIBRARY = "NCI-Nature_2016"
TOP_N = 105

SAVE_PATH = BASE_DIR / "RA2MDS_PID_FREEZE_ADAPT_TOP105"
SAVE_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

REPEATS = 80
N_SPLITS = 2
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20
TARGET_DISEASE = "MDS"

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
set_random_seed(SEED); clear_session()

print("="*100)
print(f"RA -> {TARGET_DISEASE} (MILE) | PID (NCI-Nature_2016) | TOP {TOP_N} | REPEATS={REPEATS}")
print("="*100)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def pick_col(df, cands):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in cands:
        k = str(c).strip().lower()
        if k in lookup: return lookup[k]
    return None

def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA","NAN","NONE","---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)

def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name","Gene","Genes","gene","gene_symbol","SYMBOL","X","Unnamed: 0"])
    if gene_col is None: gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))

def parse_field(value, field):
    if pd.isna(value): return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None

def map_main_label(cls):
    if cls is None or pd.isna(cls): return None
    c = str(cls).strip().upper()
    if c.startswith("AML"): return "AML"
    if c == "CLL": return "CLL"
    if c == "CML": return "CML"
    if c == "MDS": return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c: return "HE"
    if "ALL" in c: return "ALL"
    return "OTHER"

def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None: X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2/(2.0*sigma**2)).astype(np.float32)

def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0: return 1.0
    s = float(np.mean(np.sqrt(upper)))
    return s if np.isfinite(s) and s > 0 else 1.0

def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    proj = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(proj + [bias])
    out = Dense(1, activation="sigmoid", use_bias=False,
                kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model

def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model

def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]

def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, w)}

def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(spec),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }

def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2: return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values)-1)
    return (float(mean - tcrit*sem), float(mean + tcrit*sem))

def append_rows(path, rows):
    if not rows: return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)

# ------------------------------------------------------------------
# INPUT CHECK
# ------------------------------------------------------------------
print("\n[INPUT CHECK]")
for label, path in {"RA expression":RA_EXPR, "RA phenotype":RA_PHENO,
                     "MILE expression":MILE_EXPR, "MILE phenotype":MILE_FULL_PHENO,
                     "Pathway file (PID)":TOP_PATH_FILE}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists(): raise FileNotFoundError(path)

# ------------------------------------------------------------------
# LOAD RA
# ------------------------------------------------------------------
print("\n[1/7] LOADING RA")
expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")
ra_sample_col = pick_col(ph_ra, ["sample","Sample","GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw","group","label"])
if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA","HE"])].copy()
ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]
ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x=="RA" else 0 for x in ra_group.values], dtype=int)
print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full==1))} | HE={int(np.sum(y_ra_full==0))}")

# ------------------------------------------------------------------
# LOAD MILE
# ------------------------------------------------------------------
print("\n[2/7] LOADING MILE / GSE13159")
expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")
mile_sample_col = pick_col(ph_mile, ["GSM","sample","Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])
if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()
avail = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(avail)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

mds_classes = ph_mile.loc[ph_mile["main_label"]=="MDS", "leukemia_class"].value_counts()
print("\nMDS leukemia_class breakdown:")
print(mds_classes)

# ------------------------------------------------------------------
# BUILD MDS BONE-MARROW TARGET
# ------------------------------------------------------------------
print(f"\n[3/7] BUILDING {TARGET_DISEASE} BONE-MARROW TARGET")
is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)
he_bm = ph_mile.loc[(ph_mile["main_label"]=="HE") & is_bm, mile_sample_col].tolist()
case_bm = ph_mile.loc[(ph_mile["main_label"]==TARGET_DISEASE) & is_bm, mile_sample_col].tolist()
print(f"{TARGET_DISEASE} bone marrow = {len(case_bm)} | Healthy bone marrow = {len(he_bm)}")
if len(case_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid MDS/healthy bone-marrow target set found.")

target_samples_all = he_bm + case_bm
y_target_all = np.array([0]*len(he_bm) + [1]*len(case_bm), dtype=int)
print("Target total:", len(target_samples_all), "| MDS prevalence:", round(float(np.mean(y_target_all)),4))

# ------------------------------------------------------------------
# COMMON GENES
# ------------------------------------------------------------------
print("\n[4/7] COMMON GENES")
common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]
print("Common genes:", len(common_genes))

# ------------------------------------------------------------------
# LOAD TOP-105 PID PATHWAYS
# ------------------------------------------------------------------
print("\n[5/7] LOADING TOP-105 ROBUST PID (NCI-Nature_2016) PATHWAYS")
freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
print("Pathway file columns found:", list(freq.columns))
pcol = pick_col(freq, ["Pathway","Pathway_Name","Name"])
ccol = pick_col(freq, ["Selection_Count","Count","Frequency","Selection_Frequency"])
if pcol is None or ccol is None:
    raise KeyError(f"Could not detect columns. Columns present: {list(freq.columns)}")
print(f"Using pathway column = '{pcol}', count column = '{ccol}'")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()
print(f"Downloading/loading {LIBRARY}...")
pid = get_library(name=LIBRARY, organism="Human")

active_paths = []
unmatched = []
for p in top_paths:
    if p in pid:
        genes = sorted(set(pid[p]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((p, genes))
    else:
        unmatched.append(p)

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")
if unmatched:
    print(f"WARNING: {len(unmatched)} unmatched. First few: {unmatched[:5]}")
if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top105_PID_Pathways.xlsx", index=False)

# ------------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------------
completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")

# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON {TARGET_DISEASE} "
      f"({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):
    rep_number = rep + 1
    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)), test_size=RA_TEST_SIZE,
        stratify=y_ra_full, random_state=split_seed
    )
    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes, columns=source_samples
    )

    Ktr_list, pathway_sigmas = [], {}
    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))
    Xtr = np.transpose(np.stack(Ktr_list), (1,0,2)).astype(np.float32)

    clear_session(); set_random_seed(split_seed)
    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(to_inputs(Xtr), y_source, epochs=EPOCHS_RA, batch_size=BATCH_RA,
                      verbose=0, class_weight=class_weights(y_source), shuffle=True)
    source_weights = source_model.get_weights()

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):
        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list, Ktest_list = [], []
        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1,0,2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1,0,2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed*1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)
        target_model.fit(to_inputs(Xtrain_t), y_train_t, epochs=EPOCHS_ADAPT,
                          batch_size=BATCH_ADAPT, verbose=0,
                          class_weight=class_weights(y_train_t), shuffle=True)

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        met = compute_metrics(y_test_t, p_test, thr)
        met.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(met)

        for s, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": s,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session(); gc.collect()

    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session(); gc.collect()

# ------------------------------------------------------------------
# SUMMARY + SAVE
# ------------------------------------------------------------------
print("\n[7/7] SUMMARY")
metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[["AUROC","PR_AUC","Accuracy","Balanced_Accuracy",
                                              "Precision","Recall","Specificity","F1"]].mean()

summary_rows = []
for metric in ["AUROC","PR_AUC","Accuracy","Balanced_Accuracy","Precision","Recall","Specificity","F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric, "N_repeats": len(values),
        "Mean": float(np.mean(values)), "SD": float(np.std(values, ddof=1)) if len(values)>1 else np.nan,
        "Median": float(np.median(values)), "Min": float(np.min(values)), "Max": float(np.max(values)),
        "CI95_low": low, "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))
print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")

out_xlsx = SAVE_PATH / f"RA2{TARGET_DISEASE}_PID_FreezeAdapt_Top105_{REPEATS}reps.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat], "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    mds_classes.reset_index().to_excel(writer, sheet_name="MDS_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("="*100)
print("DONE.")
print("="*100)

RA -> MDS (MILE) | PID (NCI-Nature_2016) | TOP 105 | REPEATS=80
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression     : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype      : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file (PID)  : True | /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi_NCI-Nature_2016.xlsx

[1/7] LOADING RA
Reading combat_corrected_by_gse.xlsx | gene column=Gen_name
RA n=195 | RA=163 | HE=32

[2/7] LOADING MILE / GSE13159
Reading GSE13159_gene_unique.xlsx | gene column=gene

MILE label x sample type:
sample_type_norm  bone marrow  peripheral blood
main_label                                     
ALL                       710                40
AML                 

In [5]:
# ==============================================================================
# RA (SOURCE) -> ALL from MILE/GSE13159 | PID (NCI-Nature_2016) TOP-105
# LINUX GPU | SINGLE BLOCK | CHECKPOINT/RESUME
# ==============================================================================

import os, sys, gc, random, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

required = {"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn",
            "openpyxl":"openpyxl","tensorflow":"tensorflow","gseapy":"gseapy"}
for imp, pipn in required.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pipn])

import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import NonNeg
from tensorflow.keras.regularizers import l1
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed
from gseapy import get_library

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------
SEED = 42
BASE_DIR = Path("/home/altinbas-gpu/ra_leuk_project")

RA_EXPR = BASE_DIR / "combat_corrected_by_gse.xlsx"
RA_PHENO = BASE_DIR / "pheno_raw.xlsx"
MILE_EXPR = BASE_DIR / "GSE13159_gene_unique.xlsx"
MILE_FULL_PHENO = BASE_DIR / "GSE13159_FULL_PHENOTYPE.xlsx"
TOP_PATH_FILE = BASE_DIR / "yolak_secim_frekansi_NCI-Nature_2016.xlsx"

LIBRARY = "NCI-Nature_2016"
TOP_N = 105

SAVE_PATH = BASE_DIR / "RA2ALL_PID_FREEZE_ADAPT_TOP105"
SAVE_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METRICS = SAVE_PATH / "checkpoint_metrics.csv"
CHECKPOINT_PREDICTIONS = SAVE_PATH / "checkpoint_predictions.csv"

REPEATS = 80
N_SPLITS = 2
EPOCHS_RA = 400
BATCH_RA = 32
LR_RA = 0.001
EPOCHS_ADAPT = 400
BATCH_ADAPT = 32
LR_ADAPT = 1e-4
L1_VAL = 0.001
MIN_GENES = 1
RA_TEST_SIZE = 0.20
TARGET_DISEASE = "ALL"

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
set_random_seed(SEED); clear_session()

print("="*100)
print(f"RA -> {TARGET_DISEASE} (MILE) | PID (NCI-Nature_2016) | TOP {TOP_N} | REPEATS={REPEATS}")
print("="*100)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def pick_col(df, cands):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in cands:
        k = str(c).strip().lower()
        if k in lookup: return lookup[k]
    return None

def clean_expression(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    valid = (df.index != "") & (~df.index.str.upper().isin(["NA","NAN","NONE","---"]))
    df = df.loc[valid]
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    if df.isna().any().any():
        df = df.T.fillna(df.median(axis=1)).T
    return df.groupby(level=0, sort=False).mean().astype(np.float32)

def read_expr_xlsx(path):
    raw = pd.read_excel(path, engine="openpyxl")
    gene_col = pick_col(raw, ["Gen_name","Gene","Genes","gene","gene_symbol","SYMBOL","X","Unnamed: 0"])
    if gene_col is None: gene_col = raw.columns[0]
    print(f"Reading {Path(path).name} | gene column={gene_col}")
    return clean_expression(raw.set_index(gene_col))

def parse_field(value, field):
    if pd.isna(value): return None
    parts = [b.strip() for b in str(value).replace(";", "|").split("|") if b.strip()]
    for p in parts:
        if p.lower().startswith(field.lower()) and ":" in p:
            return p.split(":", 1)[1].strip()
    return None

def map_main_label(cls):
    if cls is None or pd.isna(cls): return None
    c = str(cls).strip().upper()
    if c.startswith("AML"): return "AML"
    if c == "CLL": return "CLL"
    if c == "CML": return "CML"
    if c == "MDS": return "MDS"
    if "NON-LEUKEMIA" in c or "HEALTHY" in c or "NORMAL" in c: return "HE"
    if "ALL" in c: return "ALL"
    return "OTHER"

def gaussian_kernel(X1, X2=None, sigma=1.0):
    if X2 is None: X2 = X1
    d2 = euclidean_distances(X1, X2, squared=True)
    return np.exp(-d2/(2.0*sigma**2)).astype(np.float32)

def compute_sigma(X):
    d2 = euclidean_distances(X, X, squared=True)
    upper = d2[np.triu_indices(X.shape[0], k=1)]
    upper = upper[upper > 0]
    if upper.size == 0: return 1.0
    s = float(np.mean(np.sqrt(upper)))
    return s if np.isfinite(s) and s > 0 else 1.0

def build_kernel_mlp(n_support, n_paths, lr):
    inputs = [Input(shape=(n_support,), name=f"path_in_{i}") for i in range(n_paths)]
    bias_input = Input(shape=(1,), name="bias_input")
    shared = Dense(1, use_bias=False, activation=None, name="shared_projection")
    proj = [shared(inp) for inp in inputs]
    bias = Dense(1, use_bias=False, name="bias_weight")(bias_input)
    merged = Concatenate(name="merged")(proj + [bias])
    out = Dense(1, activation="sigmoid", use_bias=False,
                kernel_regularizer=l1(L1_VAL), kernel_constraint=NonNeg(),
                name="final_output")(merged)
    model = Model(inputs=inputs + [bias_input], outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy")
    return model

def configure_target_adaptation(model):
    for layer in model.layers:
        if layer.name == "shared_projection":
            layer.trainable = True
        elif layer.name in {"final_output", "bias_weight"}:
            layer.trainable = False
        else:
            layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=LR_ADAPT), loss="binary_crossentropy")
    return model

def to_inputs(X):
    return [X[:, i, :] for i in range(X.shape[1])] + [np.ones((X.shape[0], 1), dtype=np.float32)]

def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(wt) for c, wt in zip(classes, w)}

def best_threshold_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    return {
        "AUROC": float(roc_auc_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "PR_AUC": float(average_precision_score(y, p)) if len(np.unique(y))>1 else np.nan,
        "Accuracy": float(accuracy_score(y, yhat)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, yhat)),
        "Precision": float(precision_score(y, yhat, zero_division=0)),
        "Recall": float(recall_score(y, yhat, zero_division=0)),
        "Specificity": float(spec),
        "F1": float(f1_score(y, yhat, zero_division=0)),
        "Threshold": float(thr),
    }

def ci95_t(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2: return (np.nan, np.nan)
    mean = float(np.mean(values))
    sem = st.sem(values)
    tcrit = st.t.ppf(0.975, len(values)-1)
    return (float(mean - tcrit*sem), float(mean + tcrit*sem))

def append_rows(path, rows):
    if not rows: return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)

# ------------------------------------------------------------------
# INPUT CHECK
# ------------------------------------------------------------------
print("\n[INPUT CHECK]")
for label, path in {"RA expression":RA_EXPR, "RA phenotype":RA_PHENO,
                     "MILE expression":MILE_EXPR, "MILE phenotype":MILE_FULL_PHENO,
                     "Pathway file (PID)":TOP_PATH_FILE}.items():
    print(f"{label:20s}: {path.exists()} | {path}")
    if not path.exists(): raise FileNotFoundError(path)

# ------------------------------------------------------------------
# LOAD RA
# ------------------------------------------------------------------
print("\n[1/7] LOADING RA")
expr_ra = read_expr_xlsx(RA_EXPR)
ph_ra = pd.read_excel(RA_PHENO, engine="openpyxl")
ra_sample_col = pick_col(ph_ra, ["sample","Sample","GSM"])
ra_group_col = pick_col(ph_ra, ["group_raw","group","label"])
if ra_sample_col is None or ra_group_col is None:
    raise KeyError(f"RA phenotype columns not found: {list(ph_ra.columns)}")

ph_ra[ra_sample_col] = ph_ra[ra_sample_col].astype(str).str.strip()
ph_ra[ra_group_col] = ph_ra[ra_group_col].astype(str).str.strip().str.upper()
ph_ra = ph_ra.loc[ph_ra[ra_group_col].isin(["RA","HE"])].copy()
ra_samples = [s for s in expr_ra.columns if s in set(ph_ra[ra_sample_col])]
expr_ra = expr_ra.loc[:, ra_samples]
ra_group = ph_ra.set_index(ra_sample_col).loc[ra_samples, ra_group_col]
y_ra_full = np.array([1 if x=="RA" else 0 for x in ra_group.values], dtype=int)
print(f"RA n={len(ra_samples)} | RA={int(np.sum(y_ra_full==1))} | HE={int(np.sum(y_ra_full==0))}")

# ------------------------------------------------------------------
# LOAD MILE
# ------------------------------------------------------------------
print("\n[2/7] LOADING MILE / GSE13159")
expr_mile = read_expr_xlsx(MILE_EXPR)
ph_mile = pd.read_excel(MILE_FULL_PHENO, engine="openpyxl")
mile_sample_col = pick_col(ph_mile, ["GSM","sample","Sample"])
char_col = pick_col(ph_mile, ["characteristics_ch1"])
if mile_sample_col is None or char_col is None:
    raise KeyError(f"MILE phenotype columns not found: {list(ph_mile.columns)}")

ph_mile[mile_sample_col] = ph_mile[mile_sample_col].astype(str).str.strip()
ph_mile["sample_type"] = ph_mile[char_col].apply(lambda x: parse_field(x, "sample type"))
ph_mile["leukemia_class"] = ph_mile[char_col].apply(lambda x: parse_field(x, "leukemia class"))
ph_mile["main_label"] = ph_mile["leukemia_class"].apply(map_main_label)
ph_mile["sample_type_norm"] = ph_mile["sample_type"].astype(str).str.strip().str.lower()
avail = set(expr_mile.columns)
ph_mile = ph_mile.loc[ph_mile[mile_sample_col].isin(avail)].copy()

print("\nMILE label x sample type:")
print(pd.crosstab(ph_mile["main_label"], ph_mile["sample_type_norm"]))

all_classes = ph_mile.loc[ph_mile["main_label"]=="ALL", "leukemia_class"].value_counts()
print("\nALL leukemia_class breakdown:")
print(all_classes)

# ------------------------------------------------------------------
# BUILD ALL BONE-MARROW TARGET
# ------------------------------------------------------------------
print(f"\n[3/7] BUILDING {TARGET_DISEASE} BONE-MARROW TARGET")
is_bm = ph_mile["sample_type_norm"].str.contains("bone marrow", na=False)
he_bm = ph_mile.loc[(ph_mile["main_label"]=="HE") & is_bm, mile_sample_col].tolist()
case_bm = ph_mile.loc[(ph_mile["main_label"]==TARGET_DISEASE) & is_bm, mile_sample_col].tolist()
print(f"{TARGET_DISEASE} bone marrow = {len(case_bm)} | Healthy bone marrow = {len(he_bm)}")
if len(case_bm) == 0 or len(he_bm) == 0:
    raise ValueError("No valid ALL/healthy bone-marrow target set found.")

target_samples_all = he_bm + case_bm
y_target_all = np.array([0]*len(he_bm) + [1]*len(case_bm), dtype=int)
print("Target total:", len(target_samples_all), "| ALL prevalence:", round(float(np.mean(y_target_all)),4))

# ------------------------------------------------------------------
# COMMON GENES
# ------------------------------------------------------------------
print("\n[4/7] COMMON GENES")
common_genes = sorted(set(expr_ra.index) & set(expr_mile.index))
expr_ra_c = expr_ra.loc[common_genes]
expr_mile_c = expr_mile.loc[common_genes]
print("Common genes:", len(common_genes))

# ------------------------------------------------------------------
# LOAD TOP-105 PID PATHWAYS
# ------------------------------------------------------------------
print("\n[5/7] LOADING TOP-105 ROBUST PID (NCI-Nature_2016) PATHWAYS")
freq = pd.read_excel(TOP_PATH_FILE, engine="openpyxl")
print("Pathway file columns found:", list(freq.columns))
pcol = pick_col(freq, ["Pathway","Pathway_Name","Name"])
ccol = pick_col(freq, ["Selection_Count","Count","Frequency","Selection_Frequency"])
if pcol is None or ccol is None:
    raise KeyError(f"Could not detect columns. Columns present: {list(freq.columns)}")
print(f"Using pathway column = '{pcol}', count column = '{ccol}'")

top_paths = freq.sort_values(ccol, ascending=False).head(TOP_N)[pcol].astype(str).str.strip().tolist()
print(f"Downloading/loading {LIBRARY}...")
pid = get_library(name=LIBRARY, organism="Human")

active_paths = []
unmatched = []
for p in top_paths:
    if p in pid:
        genes = sorted(set(pid[p]) & set(common_genes))
        if len(genes) >= MIN_GENES:
            active_paths.append((p, genes))
    else:
        unmatched.append(p)

print(f"Requested Top {TOP_N} -> usable pathways = {len(active_paths)}")
if unmatched:
    print(f"WARNING: {len(unmatched)} unmatched. First few: {unmatched[:5]}")
if len(active_paths) == 0:
    raise ValueError("No pathways could be mapped.")

pd.DataFrame({
    "Pathway": [p for p, _ in active_paths],
    "Common_gene_count": [len(g) for _, g in active_paths],
}).to_excel(SAVE_PATH / "Active_Top105_PID_Pathways.xlsx", index=False)

# ------------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------------
completed_repeats = set()
if CHECKPOINT_METRICS.exists():
    try:
        existing = pd.read_csv(CHECKPOINT_METRICS)
        if "Repeat" in existing.columns:
            completed_repeats = set(existing["Repeat"].astype(int).tolist())
    except Exception as e:
        print(f"WARNING: could not read existing checkpoint ({e}); starting fresh.")

if completed_repeats:
    print(f"\n[RESUME] Found checkpoint with {len(completed_repeats)} completed repeats.")
else:
    print("\n[RESUME] No checkpoint found. Starting from repeat 1.")

# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
print(f"\n[6/7] TRAINING RA -> FREEZE & ADAPT ON {TARGET_DISEASE} "
      f"({REPEATS} repeats x {N_SPLITS} folds)")

for rep in range(REPEATS):
    rep_number = rep + 1
    if rep_number in completed_repeats:
        print(f"  ... repeat {rep_number}/{REPEATS} already completed, skipping.")
        continue

    split_seed = SEED + rep

    tr_idx, _ = train_test_split(
        np.arange(len(ra_samples)), test_size=RA_TEST_SIZE,
        stratify=y_ra_full, random_state=split_seed
    )
    source_samples = np.asarray(ra_samples)[tr_idx].tolist()
    y_source = y_ra_full[tr_idx]

    ra_scaler = StandardScaler().fit(expr_ra_c[source_samples].T)
    ra_scaled = pd.DataFrame(
        ra_scaler.transform(expr_ra_c[source_samples].T).T,
        index=common_genes, columns=source_samples
    )

    Ktr_list, pathway_sigmas = [], {}
    for pathway, genes in active_paths:
        mat_tr = ra_scaled.loc[genes, source_samples].T.values
        sigma = compute_sigma(mat_tr)
        pathway_sigmas[pathway] = sigma
        Ktr_list.append(gaussian_kernel(mat_tr, sigma=sigma))
    Xtr = np.transpose(np.stack(Ktr_list), (1,0,2)).astype(np.float32)

    clear_session(); set_random_seed(split_seed)
    source_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
    source_model.fit(to_inputs(Xtr), y_source, epochs=EPOCHS_RA, batch_size=BATCH_RA,
                      verbose=0, class_weight=class_weights(y_source), shuffle=True)
    source_weights = source_model.get_weights()

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed + 29)

    repeat_rows = []
    repeat_prediction_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(target_samples_all, y_target_all), start=1
    ):
        target_train = np.asarray(target_samples_all)[train_idx].tolist()
        target_test = np.asarray(target_samples_all)[test_idx].tolist()
        y_train_t = y_target_all[train_idx]
        y_test_t = y_target_all[test_idx]

        mile_train_scaler = StandardScaler().fit(expr_mile_c[target_train].T)
        mile_train_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_train].T).T,
            index=common_genes, columns=target_train
        )
        mile_test_scaled = pd.DataFrame(
            mile_train_scaler.transform(expr_mile_c[target_test].T).T,
            index=common_genes, columns=target_test
        )

        Ktrain_list, Ktest_list = [], []
        for pathway, genes in active_paths:
            sigma = pathway_sigmas[pathway]
            source_matrix = ra_scaled.loc[genes, source_samples].T.values
            train_matrix = mile_train_scaled.loc[genes, target_train].T.values
            test_matrix = mile_test_scaled.loc[genes, target_test].T.values
            Ktrain_list.append(gaussian_kernel(train_matrix, source_matrix, sigma=sigma))
            Ktest_list.append(gaussian_kernel(test_matrix, source_matrix, sigma=sigma))

        Xtrain_t = np.transpose(np.stack(Ktrain_list), (1,0,2)).astype(np.float32)
        Xtest_t = np.transpose(np.stack(Ktest_list), (1,0,2)).astype(np.float32)

        clear_session()
        adapt_seed = split_seed*1000 + fold_idx
        set_random_seed(adapt_seed)

        target_model = build_kernel_mlp(Xtr.shape[2], Xtr.shape[1], LR_RA)
        target_model.set_weights(source_weights)
        target_model = configure_target_adaptation(target_model)
        target_model.fit(to_inputs(Xtrain_t), y_train_t, epochs=EPOCHS_ADAPT,
                          batch_size=BATCH_ADAPT, verbose=0,
                          class_weight=class_weights(y_train_t), shuffle=True)

        p_train = target_model.predict(to_inputs(Xtrain_t), verbose=0).flatten()
        thr = best_threshold_youden(y_train_t, p_train)

        p_test = target_model.predict(to_inputs(Xtest_t), verbose=0).flatten()
        met = compute_metrics(y_test_t, p_test, thr)
        met.update({"Disease": TARGET_DISEASE, "Repeat": rep_number, "Fold": fold_idx})
        repeat_rows.append(met)

        for s, y, p in zip(target_test, y_test_t, p_test):
            repeat_prediction_rows.append({
                "Repeat": rep_number, "Fold": fold_idx, "Sample": s,
                "True_label": int(y), "Predicted_probability": float(p)
            })

        del target_model, Xtrain_t, Xtest_t
        clear_session(); gc.collect()

    append_rows(CHECKPOINT_METRICS, repeat_rows)
    append_rows(CHECKPOINT_PREDICTIONS, repeat_prediction_rows)

    repeat_auroc = pd.DataFrame(repeat_rows)["AUROC"].mean()
    print(f"  ... repeat {rep_number}/{REPEATS} done | "
          f"this repeat mean AUROC = {repeat_auroc:.4f} | checkpoint saved")

    del source_model, Xtr
    clear_session(); gc.collect()

# ------------------------------------------------------------------
# SUMMARY + SAVE
# ------------------------------------------------------------------
print("\n[7/7] SUMMARY")
metrics_df = pd.read_csv(CHECKPOINT_METRICS)
predictions_df = pd.read_csv(CHECKPOINT_PREDICTIONS)

repeat_means = metrics_df.groupby("Repeat")[["AUROC","PR_AUC","Accuracy","Balanced_Accuracy",
                                              "Precision","Recall","Specificity","F1"]].mean()

summary_rows = []
for metric in ["AUROC","PR_AUC","Accuracy","Balanced_Accuracy","Precision","Recall","Specificity","F1"]:
    values = repeat_means[metric].values
    low, high = ci95_t(values)
    summary_rows.append({
        "Metric": metric, "N_repeats": len(values),
        "Mean": float(np.mean(values)), "SD": float(np.std(values, ddof=1)) if len(values)>1 else np.nan,
        "Median": float(np.median(values)), "Min": float(np.min(values)), "Max": float(np.max(values)),
        "CI95_low": low, "CI95_high": high,
    })
summary_df = pd.DataFrame(summary_rows)

auroc_values = repeat_means["AUROC"].values
try:
    wilcoxon_stat, wilcoxon_p = st.wilcoxon(auroc_values - 0.5, alternative="greater")
except Exception:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("\nSUMMARY (mean across repeats, 95% CI):")
print(summary_df.round(4).to_string(index=False))
print(f"\nOne-sided Wilcoxon test (AUROC > 0.50): statistic={wilcoxon_stat}, p={wilcoxon_p:.6g}")

out_xlsx = SAVE_PATH / f"RA2{TARGET_DISEASE}_PID_FreezeAdapt_Top105_{REPEATS}reps.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Fold_Metrics", index=False)
    repeat_means.to_excel(writer, sheet_name="Repeat_Means")
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({
        "Test": ["AUROC vs 0.50 (one-sided Wilcoxon)"],
        "Statistic": [wilcoxon_stat], "P_value": [wilcoxon_p],
        "N_repeats": [len(auroc_values)],
    }).to_excel(writer, sheet_name="Chance_Test", index=False)
    predictions_df.to_excel(writer, sheet_name="Predictions", index=False)
    all_classes.reset_index().to_excel(writer, sheet_name="ALL_Class_Check", index=False)

print("\nSaved:", out_xlsx)
print("="*100)
print("DONE.")
print("="*100)

RA -> ALL (MILE) | PID (NCI-Nature_2016) | TOP 105 | REPEATS=80
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INPUT CHECK]
RA expression       : True | /home/altinbas-gpu/ra_leuk_project/combat_corrected_by_gse.xlsx
RA phenotype        : True | /home/altinbas-gpu/ra_leuk_project/pheno_raw.xlsx
MILE expression     : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_gene_unique.xlsx
MILE phenotype      : True | /home/altinbas-gpu/ra_leuk_project/GSE13159_FULL_PHENOTYPE.xlsx
Pathway file (PID)  : True | /home/altinbas-gpu/ra_leuk_project/yolak_secim_frekansi_NCI-Nature_2016.xlsx

[1/7] LOADING RA
Reading combat_corrected_by_gse.xlsx | gene column=Gen_name
RA n=195 | RA=163 | HE=32

[2/7] LOADING MILE / GSE13159
Reading GSE13159_gene_unique.xlsx | gene column=gene

MILE label x sample type:
sample_type_norm  bone marrow  peripheral blood
main_label                                     
ALL                       710                40
AML                 